# Level 3 — geometri genellemesi, v2

Eski `train_4000_channel_map_level3.ipynb` uzerinden duzeltilmistir.

- Varsayilan: `row_phys + pattern + interaction`, `L3_RUN_TAG=interaction_fixed_v2`.
- Desen/polarizasyon/komsuluk ve CNN ayni fiziksel port sirasini kullanir.
- L3 baglaminda `level2DeltaHat` yoktur; L2 toplamdaki katkisini korur.
- Rastgele 48 Z alt kumesine sifir ortalama cezasi uygulanmaz.
- Desen normalizasyonu yalniz egitim bankalari ve egitim Z'lerinden hesaplanir.
- Eski L1/L2 ve interaction ciktilari kullanilir. Eski L3 modeli korunur.
- L3 veri hucresi sadece `z_pattern_features_physical_v2.npy` dosyasini yeniler;
  gecerli mevcut L3 satir shard'larini tekrar yazmaz. Teacher/MC uretimi yoktur.
- `covariance` istege baglidir: eklenirse RU/BR sirasiyla LL/LN/NL/NN icin 8
  analitik ozellik ayri `z_covariance_v1` shard'larina kaydedilir. Yeni MC gerekmez.
- Channel-map cikarimi yalniz `channel_map` secilirse calisir.

**Calistirma:** Kurulum hucresinde Drive yolunu kontrol et; hucreleri sirayla calistir.
Mevcut L1/L2 modellerini kullanmak icin `FORCE_RETRAIN=False` kalsin. Yeni L3
etiketi kendi modelini egitir. Kovaryans denemesi icin kurulumda `Z_INPUTS` listesine
`"covariance"` ekle ve `L3_RUN_TAG="interaction_covariance_v2"` yap.

Karsilastirma referansi: interaction newGeom RMSE=0.101109, newBoth=0.101057.
Bu dosya hazirlanirken egitim, test veya dataset uretimi calistirilmadi.


In [ ]:
ENV_FEATURES = [
    "fc", "nT1", "nT2", "nT", "nR1", "nR2", "nR", "nRIS1", "nRIS2", "nRIS",
    "K_BR", "K_RU", "isLOS_BR", "isLOS_RU", "M_BR", "M_RU", "L_BR", "L_RU",
    "dist_BR", "dist_RU",
    "c_ASA_BR", "c_ZSA_BR", "c_ASD_BR", "c_ZSD_BR", "mu_XPR_BR", "sigma_XPR_BR",
    "mu_ASA_BR", "sigma_ASA_BR", "mu_ZSA_BR", "sigma_ZSA_BR",
    "mu_ASD_BR", "sigma_ASD_BR", "mu_ZSD_BR", "sigma_ZSD_BR", "ZODoffset_BR",
    "c_ASA_RU", "c_ZSA_RU", "c_ASD_RU", "c_ZSD_RU", "mu_XPR_RU", "sigma_XPR_RU",
    "mu_ASA_RU", "sigma_ASA_RU", "mu_ZSA_RU", "sigma_ZSA_RU",
    "mu_ASD_RU", "sigma_ASD_RU", "mu_ZSD_RU", "sigma_ZSD_RU", "ZODoffset_RU",
    "eta",
]

DIRECTION_FEATURES = ["uBR_x", "uBR_y", "uBR_z", "uRU_x", "uRU_y", "uRU_z"]

SCENARIO_COLUMNS = ["scenario_BR", "scenario_RU"]

PHYSICS_FEATURES = [
    "muSNR", "sigma2Wick", "wickCV2", "traceC", "Neff", "muPwrNorm",
    "diagEnergy", "offDiagEnRatio", "lambdaMaxNorm", "spectralEntropy",
    "meanTraceRatio",
]

DIRECT_COLUMNS = [
    "w_traceUBR", "w_muBRPower", "w_nlosTraceUBR", "w_muBRShare", "w_nlosShare",
    "w_diagEnergyUBR", "w_diagPeakShareUBR", "w_diagCVUBR",
    "w_commonModeNormUBR", "w_polModeNormUBR", "w_modeBalanceUBR",
    "w_commonMeanModeNorm", "w_polMeanModeNorm",
    "w_s1Mean", "w_s1AbsMean", "w_s1PowerMean", "w_s1PeakShare", "w_s1CV",
    "w_s2Mean", "w_s2AbsMean", "w_s2PowerMean", "w_s2PeakShare", "w_s2CV",
    "w_s2ToS1PowerRatio", "w_s1s2Cosine",
]

print(f"Oznitelik listeleri: {len(ENV_FEATURES)} ortam, "
      f"{len(DIRECTION_FEATURES)} yon, {len(PHYSICS_FEATURES)} fizik, "
      f"{len(DIRECT_COLUMNS)} W fizigi")

In [ ]:
import os
import stat
import subprocess
import sys
from pathlib import Path

# ============================================================== bayraklar

FORCE_REBUILD_DATA = False
FORCE_RETRAIN = False
FORCE_REBUILD_INTERACTIONS = False
FORCE_REBUILD_CHANNEL_MAPS = False
FORCE_REBUILD_COVARIANCE = False
FORCE_REBUILD_PATTERNS = False

# L3: ilk kosu, uc temel duzeltme. Kovaryans icin listeye "covariance" ekle.
Z_INPUTS = ["row_phys", "pattern", "interaction"]
L3_RUN_TAG = "interaction_fixed_v2"
L3_USE_LEVEL2_CONTEXT = False
L3_LAMBDA_ZERO = 0.0
# L1/L2 ciktilari mevcutsa eski modelleri yeniden egitmez.


# ============================================================== yollar

REPO_URL = "https://github.com/OsmanMertYilmaz/Phy_Informed_MIMO_RIS_Env.git"
REPO_BRANCH = "codex/l3-generalization-v2"
# Ayri kod klasoru, mevcut Colab repo degisikliklerini korur.
REPO_DIR = Path("/content/Phy_Informed_MIMO_RIS_Env_l3_v2")

PROJECT = Path("/content/drive/MyDrive/Phy_Informed_MIMO_RIS_Env")

# Hiyerarsik teacher ciktilarinin kok klasorleri
HT_DIR = PROJECT / "hierarchical_teacher"
MODEL_DIR = PROJECT / "models" / "hierarchical_teacher"

L1_DIR = HT_DIR / "level1"
L2_DIR = HT_DIR / "level2"
L3_DIR = HT_DIR / "level3"
INTERACTION_DIR = HT_DIR / "z_channel_interactions_v1"
CHANNEL_MAP_DIR = HT_DIR / "z_channel_maps_v1"
COVARIANCE_DIR = HT_DIR / "z_covariance_v1"

SPLIT_FILE = HT_DIR / "ris_split.parquet"

# ============================================================== 1. Drive

from google.colab import drive, userdata

drive.mount("/content/drive")

if not PROJECT.exists():
    raise RuntimeError(f"Proje klasoru bulunamadi: {PROJECT}")

for d in [HT_DIR, MODEL_DIR, L1_DIR, L2_DIR, L3_DIR, INTERACTION_DIR, CHANNEL_MAP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Drive baglandi:", PROJECT)

# ============================================================== 2. repo

# Public repo: token gerekmez. Mevcut checkout varsa yalniz fast-forward.
if not (REPO_DIR / ".git").exists():
    if REPO_DIR.exists() and any(REPO_DIR.iterdir()):
        raise RuntimeError(f"Kod klasoru bos degil: {REPO_DIR}")
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH,
                    "--single-branch", REPO_URL, str(REPO_DIR)], check=True)
else:
    branch = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "branch", "--show-current"], text=True
    ).strip()
    if branch != REPO_BRANCH:
        raise RuntimeError(f"Beklenen dal {REPO_BRANCH}, mevcut {branch}")
    dirty = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True
    ).strip()
    if dirty:
        raise RuntimeError("L3 kod klasorunde yerel degisiklik var; otomatik guncellenmedi.")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only",
                    "origin", REPO_BRANCH], check=True)

os.chdir(REPO_DIR)

subprocess.run(["git", "log", "-1", "--oneline"], check=True)

# ============================================================== 3. import

if (REPO_DIR / "src" / "ris_env").exists():
    SRC = REPO_DIR / "src"
elif (REPO_DIR / "ris_env").exists():
    SRC = REPO_DIR
else:
    found = list(REPO_DIR.rglob("ris_env"))
    raise RuntimeError(
        "ris_env paketi bulunamadi. Bulunanlar:\n"
        + "\n".join(f"  {p}" for p in found)
    )

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# Onceki notebook'tan yuklenmis ris_env modullerini yeni kodla degistir.
for module_name in list(sys.modules):
    if module_name == "ris_env" or module_name.startswith("ris_env."):
        del sys.modules[module_name]
from ris_env.l3_features import (
    physical_ris_grid, physical_pattern_features, PATTERN_VERSION,
    COVARIANCE_VERSION, COVARIANCE_FEATURE_NAMES, covariance_component_features,
)
from ris_env.teacher_pipeline import prepare_teacher_bank
from ris_env.snr_statistics import prepare_w_state, evaluate_gamma_batch
from ris_env.ris_response import generate_ris_response_from_z

print("ris_env import edildi:", SRC)

# ============================================================== 4. ortam

import gc

import numpy as np
import pandas as pd
import torch

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print()
print("=" * 78)
print("ORTAM")
print("=" * 78)
print(f"Python  : {sys.version.split()[0]}")
print(f"numpy   : {np.__version__}")
print(f"pandas  : {pd.__version__}")
print(f"torch   : {torch.__version__}")
print(f"device  : {DEVICE}")

if DEVICE == "cuda":
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

try:
    import lightgbm as lgb
    print(f"lightgbm: {lgb.__version__}")
except ImportError:
    print("lightgbm: YOK — Level 1 hucresi kurar")

# ============================================================== 5. veri

ROOT_CANDIDATES = [
    p
    for p in PROJECT.glob("teacher_case2_varratio_*_physics_current_4000")
    if p.is_dir() and (p / "train").exists() and (p / "validation").exists()
]

if len(ROOT_CANDIDATES) != 1:
    raise RuntimeError(
        "Teacher dataset tek olarak cozulemedi:\n"
        + "\n".join(f"  {p}" for p in ROOT_CANDIDATES)
    )

ROOT = ROOT_CANDIDATES[0]

ENV_CSV = PROJECT / "environments" / "environments_4000.csv"

n_train_shards = len(list((ROOT / "train").glob("*.parquet")))
n_val_shards = len(list((ROOT / "validation").glob("*.parquet")))

print()
print("=" * 78)
print("VERI")
print("=" * 78)
print(f"Teacher dataset : {ROOT.name}")
print(f"  train shard   : {n_train_shards}")
print(f"  val shard     : {n_val_shards}")
print(f"environments    : {'VAR' if ENV_CSV.exists() else 'YOK'}  {ENV_CSV.name}")

# ---- mevcut ciktilar -------------------------------------------------------

artifacts = [
    ("ris_split.parquet", SPLIT_FILE),
    ("level1/train.csv", L1_DIR / "train.csv"),
    ("level1/validation.csv", L1_DIR / "validation.csv"),
    ("level2/train.csv", L2_DIR / "train.csv"),
    ("level2/validation.csv", L2_DIR / "validation.csv"),
    ("level3/build_manifest.json", L3_DIR / "build_manifest.json"),
    ("z_channel_interactions_v1", INTERACTION_DIR / "manifest.json"),
    ("z_channel_maps_v1", CHANNEL_MAP_DIR / "manifest.json"),
]

print()
print("Mevcut ciktilar:")

for label, path in artifacts:
    if path.exists():
        size = path.stat().st_size

        unit = f"{size/1e6:.1f} MB" if size >= 1e6 else f"{size/1e3:.0f} KB"

        print(f"  [VAR] {label:<28} {unit}")
    else:
        print(f"  [YOK] {label}")

# ============================================================== 6. ozet

print()
print("=" * 78)
print("BAYRAKLAR")
print("=" * 78)
print(f"  FORCE_REBUILD_DATA = {FORCE_REBUILD_DATA}")
print(f"  FORCE_RETRAIN      = {FORCE_RETRAIN}")
print(f"  FORCE_REBUILD_INTERACTIONS = {FORCE_REBUILD_INTERACTIONS}")
print(f"  FORCE_REBUILD_CHANNEL_MAPS = {FORCE_REBUILD_CHANNEL_MAPS}")
print()
print("Asagi hucreler bu bayraklari okur. Veri setleri ve modeller varsa")
print("yeniden uretilmez; zorlamak icin ilgili bayragi True yap ve bu")
print("hucreyi tekrar kos.")

print()
print("KURULUM TAMAM")


In [ ]:
import shutil
import time

LOCAL_ROOT = Path("/content/teacher_data")

DRIVE_ROOT = ROOT   # Drive'daki orijinali sakla

if LOCAL_ROOT.exists():
    ROOT = LOCAL_ROOT
    n = sum(1 for _ in LOCAL_ROOT.rglob("*.parquet"))
    print(f"Yerel kopya mevcut: {LOCAL_ROOT}  ({n} parquet)")

else:
    files = sorted(DRIVE_ROOT.rglob("*"))
    files = [f for f in files if f.is_file()]

    total = sum(f.stat().st_size for f in files)

    print(f"Kopyalaniyor: {len(files)} dosya, {total/1e9:.2f} GB")
    print(f"  {DRIVE_ROOT}")
    print(f"  -> {LOCAL_ROOT}")

    done = 0
    t0 = time.time()
    last = 0.0

    try:
        for i, src in enumerate(files, 1):
            dst = LOCAL_ROOT / src.relative_to(DRIVE_ROOT)
            dst.parent.mkdir(parents=True, exist_ok=True)

            for attempt in range(4):
                try:
                    shutil.copy2(src, dst)
                    break
                except Exception as exc:
                    if attempt == 3:
                        raise

                    print(f"\n  {src.name} kopyalanamadi "
                          f"({type(exc).__name__}), yeniden baglaniyor...")

                    time.sleep(5 * (attempt + 1))

                    from google.colab import drive
                    drive.mount("/content/drive", force_remount=True)

            done += src.stat().st_size

            pct = 100 * done / total

            if pct - last >= 1.0 or i == len(files):
                elapsed = time.time() - t0
                rate = done / max(elapsed, 1e-9)
                eta = (total - done) / max(rate, 1e-9)

                bar = "#" * int(pct / 2.5) + "." * (40 - int(pct / 2.5))

                print(
                    f"\r  [{bar}] {pct:5.1f}%  "
                    f"{done/1e9:5.2f}/{total/1e9:.2f} GB  "
                    f"{rate/1e6:4.0f} MB/s  kalan {eta/60:4.1f} dk",
                    end="",
                    flush=True,
                )

                last = pct

    except BaseException:
        print("\n\nKopyalama yarida kesildi. Eksik kopya kullanilmasin diye")
        print(f"su klasoru silmelisin: {LOCAL_ROOT}")
        raise

    print(f"\n\nTamamlandi: {(time.time()-t0)/60:.1f} dk")

    ROOT = LOCAL_ROOT

# Disk kontrolu
import subprocess
subprocess.run(["df", "-h", "/content"], check=False)

print(f"\nROOT = {ROOT}")

In [ ]:
import json
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# ---- hucre 1'den gelenler --------------------------------------------------

for name in ["ROOT", "HT_DIR", "L1_DIR", "SPLIT_FILE", "FORCE_REBUILD_DATA"]:
    if name not in globals():
        raise RuntimeError(f"{name} tanimli degil — once kurulum hucresini kos.")

TRAIN_OUT = L1_DIR / "train.csv"
VAL_OUT = L1_DIR / "validation.csv"
SETTINGS_FILE = HT_DIR / "settings.json"

# ============================================================== sabitler

N_Z_TOTAL = 512
N_Z_TRAIN = 358
N_Z_HOLDOUT = 154
SPLIT_SEED = 42073

EXPECTED_TRAIN_BANKS = 2800
EXPECTED_VAL_BANKS = 600

# ============================================================== oznitelikler

GEOMETRY_COLUMNS = [
    "gnb_x", "gnb_y", "gnb_z",
    "ris_x", "ris_y", "ris_z",
    "ue_x", "ue_y", "ue_z",
]

# Tek sorguda hesaplanabilen girdiler
LSP_INPUTS = ENV_FEATURES + DIRECTION_FEATURES

PHYSICS_AGG_FEATURES = [
    f"{b}_{s}" for b in PHYSICS_FEATURES for s in ("mean", "std")
]

# Bank baglami gerektirenler (ablasyon icin)
CONTEXT_INPUTS = PHYSICS_AGG_FEATURES

LABEL_COLUMNS = ["targetBankMean", "targetBankStd", "nValidConfig"]

assert len(ENV_FEATURES) == 51
assert len(DIRECTION_FEATURES) == 6
assert len(PHYSICS_AGG_FEATURES) == 22
assert len(LSP_INPUTS) == 57

OUTPUT_COLUMNS = (
    ["bankID", *SCENARIO_COLUMNS, *LSP_INPUTS, *CONTEXT_INPUTS, *LABEL_COLUMNS]
)

READ_COLUMNS = [
    "bankID", "zCandidate", "analyticValid", "targetLogVarRatio",
    *SCENARIO_COLUMNS, *ENV_FEATURES, *GEOMETRY_COLUMNS, *PHYSICS_FEATURES,
]

# ============================================================== onbellek

cached = TRAIN_OUT.exists() and VAL_OUT.exists() and not FORCE_REBUILD_DATA

if cached:
    level1_train = pd.read_csv(TRAIN_OUT)
    level1_val = pd.read_csv(VAL_OUT)

    for label, D in [("train", level1_train), ("validation", level1_val)]:
        missing = set(OUTPUT_COLUMNS) - set(D.columns)

        if missing:
            print(f"{label}.csv eksik sutun iceriyor, yeniden uretilecek: "
                  f"{sorted(missing)[:5]}")
            cached = False
            break

if cached:
    print("=" * 78)
    print("LEVEL 1 VERI SETI MEVCUT — yeniden uretilmedi")
    print("=" * 78)
    print(f"  train      : {len(level1_train):,} bank")
    print(f"  validation : {len(level1_val):,} bank")
    print()
    print("Bastan kurmak icin kurulum hucresinde FORCE_REBUILD_DATA = True")

else:
    t0 = time.time()

    print("=" * 78)
    print("LEVEL 1 VERI SETI KURULUYOR")
    print("=" * 78)

    # ------------------------------------------------------ bank kimlikleri

    def collect_banks(split_name, expected):
        files = sorted((ROOT / split_name).glob("*.parquet"))

        if not files:
            raise RuntimeError(f"{split_name} icin parquet yok")

        parts = [
            pq.read_table(
                p, columns=["bankID"], filters=[("zCandidate", "=", 0)]
            ).column("bankID").to_numpy(zero_copy_only=False)
            for p in files
        ]

        banks = np.unique(np.concatenate(parts)).astype(np.int64)

        if len(banks) != expected:
            raise RuntimeError(f"{split_name}: {len(banks)} bank, {expected} bekleniyor")

        return banks

    train_banks = collect_banks("train", EXPECTED_TRAIN_BANKS)
    val_banks = collect_banks("validation", EXPECTED_VAL_BANKS)

    if np.intersect1d(train_banks, val_banks).size:
        raise RuntimeError("Egitim ve dogrulama banklari ortusuyor")

    print(f"Bank : {len(train_banks):,} egitim / {len(val_banks):,} dogrulama")

    # ------------------------------------------------------ z bolunmesi

    if SPLIT_FILE.exists():
        split = pd.read_parquet(SPLIT_FILE)
        print(f"z bolunmesi mevcut: {SPLIT_FILE.name}")

    else:
        rng = np.random.default_rng(SPLIT_SEED)

        order = rng.permutation(N_Z_TOTAL)

        global_train = np.zeros(N_Z_TOTAL, dtype=bool)
        global_train[order[:N_Z_TRAIN]] = True

        all_banks = np.concatenate([train_banks, val_banks])

        split = pd.DataFrame(
            {
                "bankID": np.repeat(all_banks, N_Z_TOTAL),
                "zCandidate": np.tile(
                    np.arange(N_Z_TOTAL, dtype=np.int16), len(all_banks)
                ),
                "isTrainZ": np.tile(global_train, len(all_banks)),
            }
        )

        split.to_parquet(SPLIT_FILE, index=False)

        with open(SETTINGS_FILE, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "splitSeed": SPLIT_SEED,
                    "zTotal": N_Z_TOTAL,
                    "zTrain": N_Z_TRAIN,
                    "zHoldout": N_Z_HOLDOUT,
                },
                f,
                indent=2,
            )

        print(f"z bolunmesi uretildi (seed {SPLIT_SEED})")

    # ---- bolunme global olmali ---------------------------------------

    pivot = (
        split.pivot(index="bankID", columns="zCandidate", values="isTrainZ")
        .sort_index()
        .sort_index(axis=1)
        .to_numpy(dtype=bool)
    )

    IS_TRAIN_Z = pivot[0]

    if not np.all(pivot == IS_TRAIN_Z[None, :]):
        raise RuntimeError("z bolunmesi global degil")

    if IS_TRAIN_Z.sum() != N_Z_TRAIN:
        raise RuntimeError(f"Egitim z sayisi {IS_TRAIN_Z.sum()}, {N_Z_TRAIN} bekleniyor")

    if pivot.shape[0] != len(train_banks) + len(val_banks):
        raise RuntimeError("Bolunmedeki bank sayisi uyusmuyor")

    del pivot

    print(f"z : {int(IS_TRAIN_Z.sum())} egitim / "
          f"{int((~IS_TRAIN_Z).sum())} holdout  (global, tum banklar ayni)")

    # ------------------------------------------------------ toplama

    n_phys = len(PHYSICS_FEATURES)

    def build_split(split_name, banks):
        files = sorted((ROOT / split_name).glob("*.parquet"))

        index = {int(b): i for i, b in enumerate(banks)}
        n = len(banks)

        p_sum = np.zeros((n, n_phys))
        p_sq = np.zeros((n, n_phys))
        y_sum = np.zeros(n)
        y_sq = np.zeros(n)
        count = np.zeros(n, dtype=np.int64)

        static = {}

        for i, path in enumerate(files, 1):
            D = pq.read_table(
                path, columns=READ_COLUMNS, use_threads=True
            ).to_pandas(split_blocks=True)

            D = D.loc[D["analyticValid"].to_numpy(dtype=bool)]

            if len(D) == 0:
                continue

            z = D["zCandidate"].to_numpy(np.int64)

            if z.min() < 0 or z.max() >= N_Z_TOTAL:
                raise RuntimeError("zCandidate araligi bozuk")

            D = D.loc[IS_TRAIN_Z[z]]

            if len(D) == 0:
                continue

            bank = D["bankID"].to_numpy(np.int64)

            unknown = set(np.unique(bank).tolist()) - set(index)

            if unknown:
                raise RuntimeError(f"Beklenmeyen bank: {sorted(unknown)[:5]}")

            row = np.array([index[int(b)] for b in bank])

            y = D["targetLogVarRatio"].to_numpy(np.float64)
            P = D[PHYSICS_FEATURES].to_numpy(np.float64)

            if not np.isfinite(y).all() or not np.isfinite(P).all():
                raise RuntimeError(f"{path.name}: NaN/Inf")

            np.add.at(p_sum, row, P)
            np.add.at(p_sq, row, P * P)
            np.add.at(y_sum, row, y)
            np.add.at(y_sq, row, y * y)
            np.add.at(count, row, 1)

            for _, r in D.drop_duplicates("bankID").iterrows():
                b = int(r["bankID"])

                if b not in static:
                    static[b] = r

            if i % 60 == 0 or i == len(files):
                print(f"  [{split_name}] {i}/{len(files)} shard  "
                      f"[{time.time()-t0:.0f}s]")

        if (count == 0).any():
            raise RuntimeError(
                f"{split_name}: {int((count==0).sum())} bankta gecerli satir yok"
            )

        if len(static) != n:
            raise RuntimeError(f"{split_name}: {n - len(static)} bank sabiti eksik")

        p_mean = p_sum / count[:, None]
        p_std = np.sqrt(np.maximum(p_sq / count[:, None] - p_mean**2, 0.0))

        y_mean = y_sum / count
        y_std = np.sqrt(np.maximum(y_sq / count - y_mean**2, 0.0))

        S = pd.DataFrame([static[int(b)] for b in banks]).reset_index(drop=True)

        out = pd.DataFrame({"bankID": banks.astype(np.int64)})

        for col in SCENARIO_COLUMNS:
            out[col] = S[col].astype(str).to_numpy()

        for col in ENV_FEATURES:
            out[col] = S[col].to_numpy()

        def unit(ax, bx, ay, by, az, bz):
            dx = S[ax].to_numpy(np.float64) - S[bx].to_numpy(np.float64)
            dy = S[ay].to_numpy(np.float64) - S[by].to_numpy(np.float64)
            dz = S[az].to_numpy(np.float64) - S[bz].to_numpy(np.float64)

            d = np.sqrt(dx**2 + dy**2 + dz**2)

            if (d <= 0).any():
                raise RuntimeError("Sifir uzunluklu yon vektoru")

            return dx / d, dy / d, dz / d

        out["uBR_x"], out["uBR_y"], out["uBR_z"] = unit(
            "ris_x", "gnb_x", "ris_y", "gnb_y", "ris_z", "gnb_z"
        )

        out["uRU_x"], out["uRU_y"], out["uRU_z"] = unit(
            "ue_x", "ris_x", "ue_y", "ris_y", "ue_z", "ris_z"
        )

        for j, feature in enumerate(PHYSICS_FEATURES):
            out[f"{feature}_mean"] = p_mean[:, j]
            out[f"{feature}_std"] = p_std[:, j]

        out["targetBankMean"] = y_mean
        out["targetBankStd"] = y_std
        out["nValidConfig"] = count

        return out[OUTPUT_COLUMNS]

    level1_train = build_split("train", train_banks)
    level1_val = build_split("validation", val_banks)

    for label, D in [("train", level1_train), ("validation", level1_val)]:
        cols = LSP_INPUTS + CONTEXT_INPUTS

        if D[cols].isna().any().any():
            bad = D[cols].columns[D[cols].isna().any()].tolist()
            raise RuntimeError(f"{label}: NaN iceren sutunlar {bad}")

    level1_train.to_csv(TRAIN_OUT, index=False)
    level1_val.to_csv(VAL_OUT, index=False)

    print()
    print(f"Yazildi [{(time.time()-t0)/60:.1f} dk]")
    print(f"  {TRAIN_OUT}")
    print(f"  {VAL_OUT}")

# ============================================================== ozet

print()
print("=" * 78)
print("SUTUN SINIFLANDIRMASI")
print("=" * 78)
print(f"ANA GIRDI (LSP)      : {len(LSP_INPUTS):>3}   "
      f"{len(ENV_FEATURES)} ortam + {len(DIRECTION_FEATURES)} yon")
print(f"  -> tek (LSP, W, z) sorgusunda hepsi hesaplanabilir")
print()
print(f"ABLASYON GIRDISI     : {len(CONTEXT_INPUTS):>3}   fizik agregasi")
print(f"  -> bankin 11456 konfigurasyonunu gerektirir, varsayilan KAPALI")
print()
print(f"Senaryo (kategorik)  : {len(SCENARIO_COLUMNS):>3}")
print(f"Etiket               : {len(LABEL_COLUMNS):>3}   ASLA GIRDI OLMAZ")

print()
print("HEDEF")

for label, D in [("train", level1_train), ("validation", level1_val)]:
    y = D["targetBankMean"]
    s = D["targetBankStd"]

    print(f"  {label:<11} targetBankMean  mean={y.mean():+.6f}  "
          f"std={y.std(ddof=0):.6f}  [{y.min():+.3f}, {y.max():+.3f}]")
    print(f"  {'':<11} targetBankStd   mean={s.mean():+.6f}  "
          f"std={s.std(ddof=0):.6f}")

c = level1_train["nValidConfig"]

print()
print(f"Gecerli konfigurasyon / bank : min {c.min()}, "
      f"medyan {int(c.median())}, max {c.max()}")
print(f"  (32 W x {N_Z_TRAIN} z = {32 * N_Z_TRAIN} tam dolu grup)")

if c.min() < 32 * N_Z_TRAIN:
    n_partial = int((c < 32 * N_Z_TRAIN).sum())
    print(f"  {n_partial} bankta bazi (bank, W) gruplari analytic gecersiz")

In [ ]:
import json
import time

import numpy as np
import pandas as pd

try:
    import lightgbm as lgb
except ImportError:
    import subprocess
    import sys

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "lightgbm"], check=True
    )
    import lightgbm as lgb

from sklearn.model_selection import KFold

for name in ["L1_DIR", "MODEL_DIR", "FORCE_RETRAIN"]:
    if name not in globals():
        raise RuntimeError(f"{name} tanimli degil — once kurulum hucresini kos.")

# ============================================================== ayarlar

USE_PHYSICS = False      # True -> 78 oznitelik (ablasyon)
FIT_STD_HEAD = True      # ikinci GBM: targetBankStd

SEED = 42

N_FOLDS = 5
N_REPEATS = 5

RUN_SEARCH = False
N_SEARCH = 24
INNER_VAL_FRACTION = 0.10
ROUND_INFLATION = 1.10

COMPARE_SINGLE_MODEL = True   # deployment icin tek model vs topluluk

# Aramada kazanan bolge: sig agac, yuksek lr, guclu L2
PARAMS = {
    "learning_rate": 0.03,
    "num_leaves": 31,
    "min_data_in_leaf": 10,
    "feature_fraction": 0.65,
    "bagging_fraction": 1.0,
    "lambda_l2": 1.0,
    "lambda_l1": 0.0,
}

FIXED_ROUNDS = 900

BASE_PARAMS = {
    "objective": "regression",
    "metric": "rmse",
    "verbosity": -1,
    "num_threads": 4,
    "bagging_freq": 1,
}

VARIANT = "phys" if USE_PHYSICS else "lsp"

MODEL_ROOT = MODEL_DIR / f"level1_gbm_{VARIANT}"
BOOSTER_DIR = MODEL_ROOT / "boosters"

MODEL_ROOT.mkdir(parents=True, exist_ok=True)
BOOSTER_DIR.mkdir(parents=True, exist_ok=True)

print("LightGBM :", lgb.__version__)
print("Varyant  :", "LSP + FIZIK (78)" if USE_PHYSICS else "SALT LSP (56)")

# ============================================================== veri

train_df = pd.read_csv(L1_DIR / "train.csv")
val_df = pd.read_csv(L1_DIR / "validation.csv")

assert len(train_df) == 2800, len(train_df)
assert len(val_df) == 600, len(val_df)

if np.intersect1d(train_df["bankID"], val_df["bankID"]).size:
    raise RuntimeError("Egitim ve dogrulama banklari ortusuyor")

PHYSICS_AGG = [f"{b}_{s}" for b in PHYSICS_FEATURES for s in ("mean", "std")]

# BEYAZ LISTE — kara liste kullanmak Level 3'te targetWStd'nin sizmasina
# yol acmisti
NUMERIC_FEATURES = ENV_FEATURES + DIRECTION_FEATURES

if USE_PHYSICS:
    NUMERIC_FEATURES = NUMERIC_FEATURES + PHYSICS_AGG

missing = set(NUMERIC_FEATURES) - set(train_df.columns)

if missing:
    raise RuntimeError(f"Eksik sutun: {sorted(missing)}")

assert len(NUMERIC_FEATURES) == (79 if USE_PHYSICS else 57)

ALL_FEATURES = NUMERIC_FEATURES + SCENARIO_COLUMNS

TARGETS = ["targetBankMean"] + (["targetBankStd"] if FIT_STD_HEAD else [])

SCENARIO_VALUES = sorted(
    set(train_df["scenario_BR"].astype(str)) | set(train_df["scenario_RU"].astype(str))
)

SCENARIO_MAP = {s: i for i, s in enumerate(SCENARIO_VALUES)}
SCENARIO_CODES = sorted(SCENARIO_MAP.values())

print()
print(f"Oznitelik : {len(NUMERIC_FEATURES)} sayisal + {len(SCENARIO_COLUMNS)} senaryo")
print(f"Senaryo   : {len(SCENARIO_MAP)}")
print(f"Hedef     : {', '.join(TARGETS)}")


def build_frame(df, rows):
    X = df.iloc[rows][NUMERIC_FEATURES].copy().reset_index(drop=True)

    for col in SCENARIO_COLUMNS:
        codes = (
            df.iloc[rows][col].astype(str).map(SCENARIO_MAP)
            .fillna(-1).astype("int32").to_numpy()
        )

        X[col] = pd.Categorical(codes, categories=SCENARIO_CODES)

    return X


def rmse(pred, true):
    return float(np.sqrt(np.mean((pred - true) ** 2)))


def report(pred, true):
    err = pred - true
    return {
        "RMSE": float(np.sqrt(np.mean(err**2))),
        "MAE": float(np.mean(np.abs(err))),
        "R2": float(1.0 - np.sum(err**2) / np.sum((true - true.mean()) ** 2)),
        "bias": float(np.mean(err)),
    }


ALL_ROWS = np.arange(len(train_df))
VAL_ROWS = np.arange(len(val_df))

X_val_frame = build_frame(val_df, VAL_ROWS)

# ============================================================== arama


def cv_evaluate(params, y, seed=SEED):
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    rng = np.random.default_rng(seed)

    oof = np.full(len(train_df), np.nan)
    rounds = []

    for fit_all, oof_idx in kf.split(ALL_ROWS):
        shuffled = rng.permutation(fit_all)
        n_stop = int(round(INNER_VAL_FRACTION * len(shuffled)))

        stop_idx, fit_idx = shuffled[:n_stop], shuffled[n_stop:]

        p = {**BASE_PARAMS, **params, "seed": seed}

        dfit = lgb.Dataset(
            build_frame(train_df, fit_idx),
            label=y[fit_idx],
            categorical_feature=SCENARIO_COLUMNS,
        )

        dstop = lgb.Dataset(
            build_frame(train_df, stop_idx), label=y[stop_idx], reference=dfit
        )

        b = lgb.train(
            p, dfit, num_boost_round=6000, valid_sets=[dstop],
            callbacks=[lgb.early_stopping(200, verbose=False)],
        )

        oof[oof_idx] = b.predict(
            build_frame(train_df, oof_idx), num_iteration=b.best_iteration
        )

        rounds.append(b.best_iteration)

    return rmse(oof, y), float(np.mean(rounds))


if RUN_SEARCH:
    SEARCH_SPACE = {
        "learning_rate": [0.015, 0.02, 0.03, 0.05],
        "num_leaves": [15, 31, 63, 127],
        "min_data_in_leaf": [5, 10, 20, 40],
        "feature_fraction": [0.5, 0.65, 0.8, 1.0],
        "bagging_fraction": [0.6, 0.8, 1.0],
        "lambda_l2": [0.0, 1.0, 5.0, 20.0],
        "lambda_l1": [0.0, 0.5, 2.0],
    }

    print()
    print("#" * 78)
    print(f"ARAMA ({N_SEARCH} yapilandirma, sadece egitim banklarinda)")
    print("#" * 78)

    y0 = train_df["targetBankMean"].to_numpy(np.float64)

    r0, rounds0 = cv_evaluate(PARAMS, y0)

    print(f"mevcut ayar : CV={r0:.6f} tur={rounds0:.0f}")

    srng = np.random.default_rng(SEED)

    results = [{"tag": "current", "params": dict(PARAMS), "rmse": r0, "rounds": rounds0}]

    for i in range(1, N_SEARCH + 1):
        cand = {k: v[srng.integers(len(v))] for k, v in SEARCH_SPACE.items()}
        cand = {
            k: (float(v) if isinstance(v, (float, np.floating)) else int(v))
            for k, v in cand.items()
        }

        r, rd = cv_evaluate(cand, y0)

        results.append({"tag": f"c{i:02d}", "params": cand, "rmse": r, "rounds": rd})

        mark = " *" if r < min(x["rmse"] for x in results[:-1]) else ""

        print(f"[{i:02d}] CV={r:.6f} tur={rd:>5.0f} {json.dumps(cand)}{mark}")

    results.sort(key=lambda x: x["rmse"])

    PARAMS = results[0]["params"]
    FIXED_ROUNDS = int(round(results[0]["rounds"] * ROUND_INFLATION))

    print()
    print("EN IYI:", results[0]["tag"], json.dumps(PARAMS))
    print(f"CV {results[0]['rmse']:.6f} (mevcut {r0:.6f}, "
          f"kazanc {r0 - results[0]['rmse']:+.6f})")
    print()
    print("UYARI: CV skorunun kendi gurultusu ~0.007. Kazanc bu mertebedeyse")
    print("oldugu gibi raporlanmamali — 24 aday arasindan secim yapildi.")

# ============================================================== egitim

META_FILE = MODEL_ROOT / "metadata.json"
PRED_FILE = MODEL_ROOT / "predictions.csv"

cached = META_FILE.exists() and PRED_FILE.exists() and not FORCE_RETRAIN

if cached:
    with open(META_FILE, encoding="utf-8") as f:
        metadata = json.load(f)

    if metadata.get("features") != ALL_FEATURES or metadata.get("targets") != TARGETS:
        print("\nKaydedilmis yapilandirma farkli — yeniden egitiliyor.")
        cached = False

if cached:
    preds = pd.read_csv(PRED_FILE)

    print()
    print("=" * 78)
    print("MEVCUT MODEL KULLANILIYOR — yeniden egitilmedi")
    print("=" * 78)
    print(f"  {MODEL_ROOT}")
    print("  Yeniden egitmek icin kurulum hucresinde FORCE_RETRAIN = True")

else:
    t0 = time.time()

    print()
    print("#" * 78)
    print(f"TEKRARLI K-FOLD ({N_REPEATS} x {N_FOLDS}, sabit {FIXED_ROUNDS} tur)")
    print("#" * 78)

    store = {}
    fold_records = []

    for target in TARGETS:
        y = train_df[target].to_numpy(np.float64)

        oof_sum = np.zeros(len(train_df))
        oof_count = np.zeros(len(train_df))
        val_parts = []
        gains = []

        for repeat in range(N_REPEATS):
            kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED + 1000 * repeat)

            for fold, (fit_idx, oof_idx) in enumerate(kf.split(ALL_ROWS), 1):
                p = {**BASE_PARAMS, **PARAMS, "seed": SEED + 1000 * repeat + fold}

                dfit = lgb.Dataset(
                    build_frame(train_df, fit_idx),
                    label=y[fit_idx],
                    categorical_feature=SCENARIO_COLUMNS,
                )

                b = lgb.train(p, dfit, num_boost_round=FIXED_ROUNDS)

                pred = b.predict(build_frame(train_df, oof_idx))

                oof_sum[oof_idx] += pred
                oof_count[oof_idx] += 1

                val_parts.append(b.predict(X_val_frame))

                gains.append(b.feature_importance(importance_type="gain"))

                b.save_model(str(BOOSTER_DIR / f"{target}_r{repeat}_f{fold}.txt"))

                fold_records.append(
                    {
                        "target": target,
                        "repeat": repeat,
                        "fold": fold,
                        "foldRMSE": rmse(pred, y[oof_idx]),
                    }
                )

        assert np.all(oof_count == N_REPEATS)

        store[target] = {
            "oof": oof_sum / oof_count,
            "val": np.mean(val_parts, axis=0),
            "gain": np.mean(gains, axis=0),
            "featureNames": b.feature_name(),
        }

        print(f"  {target}: {N_REPEATS * N_FOLDS} model  [{time.time()-t0:.0f}s]")

    folds = pd.DataFrame(fold_records)

    # ---- kayit -----------------------------------------------------------

    preds = pd.concat(
        [
            pd.DataFrame(
                {
                    "bankID": train_df["bankID"].to_numpy(np.int64),
                    "split": "train",
                    **{f"true_{t}": train_df[t].to_numpy() for t in TARGETS},
                    **{f"hat_{t}": store[t]["oof"] for t in TARGETS},
                }
            ),
            pd.DataFrame(
                {
                    "bankID": val_df["bankID"].to_numpy(np.int64),
                    "split": "validation",
                    **{f"true_{t}": val_df[t].to_numpy() for t in TARGETS},
                    **{f"hat_{t}": store[t]["val"] for t in TARGETS},
                }
            ),
        ],
        ignore_index=True,
    )

    preds.to_csv(PRED_FILE, index=False)
    folds.to_csv(MODEL_ROOT / "fold_scores.csv", index=False)

    metadata = {
        "variant": VARIANT,
        "usePhysics": USE_PHYSICS,
        "features": ALL_FEATURES,
        "targets": TARGETS,
        "params": PARAMS,
        "fixedRounds": FIXED_ROUNDS,
        "nFolds": N_FOLDS,
        "nRepeats": N_REPEATS,
        "seed": SEED,
        "scenarioMap": SCENARIO_MAP,
        "featureImportance": {
            t: (
                pd.DataFrame(
                    {"feature": store[t]["featureNames"], "gain": store[t]["gain"]}
                )
                .sort_values("gain", ascending=False)
                .head(30)
                .to_dict("records")
            )
            for t in TARGETS
        },
    }

    with open(META_FILE, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    print(f"\nKaydedildi: {MODEL_ROOT}  [{(time.time()-t0)/60:.1f} dk]")

# ============================================================== sonuclar

is_train = preds["split"].to_numpy() == "train"

print()
print("#" * 78)
print("SONUCLAR")
print("#" * 78)

for target in TARGETS:
    oof_m = report(
        preds.loc[is_train, f"hat_{target}"].to_numpy(),
        preds.loc[is_train, f"true_{target}"].to_numpy(),
    )

    val_m = report(
        preds.loc[~is_train, f"hat_{target}"].to_numpy(),
        preds.loc[~is_train, f"true_{target}"].to_numpy(),
    )

    print()
    print(f"--- {target} ---")
    print(f"  OOF  ({int(is_train.sum())} bank, {N_REPEATS} model)   "
          f"RMSE={oof_m['RMSE']:.6f}  R2={oof_m['R2']:.4f}")
    print(f"  VAL  ({int((~is_train).sum())} bank, {N_FOLDS*N_REPEATS} model)  "
          f"RMSE={val_m['RMSE']:.6f}  R2={val_m['R2']:.4f}  "
          f"bias={val_m['bias']:+.3e}")
    print(f"  OOF/VAL orani : {oof_m['RMSE'] / val_m['RMSE']:.3f}")

    if target == "targetBankMean":
        MAIN_VAL = val_m
        MAIN_OOF = oof_m

se = MAIN_VAL["RMSE"] / np.sqrt(2 * int((~is_train).sum()))

print()
print(f"Val RMSE belirsizligi (~1 sigma): +/-{se:.4f}")
print(f"{2*se:.3f} altindaki farklari gurultu say.")

print()
print("REFERANS (targetBankMean, dogrulama)")
print("  salt LSP  (56) : 0.111289")
print("  + fizik   (78) : 0.103794")
print(f"  bu kosu   ({len(NUMERIC_FEATURES)}) : {MAIN_VAL['RMSE']:.6f}")

# ---- onem siralamasi ------------------------------------------------------

if not cached:
    print()
    print("EN ONEMLI 12 OZNITELIK (targetBankMean)")

    imp = (
        pd.DataFrame(
            {
                "feature": store["targetBankMean"]["featureNames"],
                "gain": store["targetBankMean"]["gain"],
            }
        )
        .sort_values("gain", ascending=False)
        .reset_index(drop=True)
    )

    imp["share"] = 100 * imp["gain"] / imp["gain"].sum()

    for _, r in imp.head(12).iterrows():
        print(f"  {r['feature']:<26}{r['share']:>7.2f}%")

# ============================================================== tek model

if COMPARE_SINGLE_MODEL and not cached:
    print()
    print("=" * 78)
    print("DEPLOYMENT: TEK MODEL vs TOPLULUK")
    print("=" * 78)

    y = train_df["targetBankMean"].to_numpy(np.float64)

    p = {**BASE_PARAMS, **PARAMS, "seed": SEED}

    dall = lgb.Dataset(
        build_frame(train_df, ALL_ROWS), label=y, categorical_feature=SCENARIO_COLUMNS
    )

    single = lgb.train(p, dall, num_boost_round=FIXED_ROUNDS)

    single.save_model(str(MODEL_ROOT / "single_full.txt"))

    single_val = single.predict(X_val_frame)

    single_m = report(single_val, val_df["targetBankMean"].to_numpy(np.float64))

    print(f"  Tek model (tum veri, {FIXED_ROUNDS} tur) : {single_m['RMSE']:.6f}")
    print(f"  Topluluk ({N_FOLDS*N_REPEATS} model)          : {MAIN_VAL['RMSE']:.6f}")
    print(f"  Fark                                : "
          f"{single_m['RMSE'] - MAIN_VAL['RMSE']:+.6f}")
    print()

    if single_m["RMSE"] - MAIN_VAL["RMSE"] < 2 * se:
        print("  Tek model toplulukla gurultu icinde esit. Deployment'ta tek")
        print("  model tasimak yeterli; single_full.txt kullanilabilir.")
    else:
        print("  Topluluk belirgin sekilde daha iyi. Deployment'ta 25 booster'in")
        print("  HEPSI yuklenip ortalanmali.")

    print()
    print("  NOT: level1Hat uretimi icin YINE out-of-fold gerekli. Tek model")
    print("  egitim banklarini gordugu icin Level 2'ye veremezsin.")

print()
print("SIRADAKI ADIM: Level 2 veri seti hucresi")
print(f"  level1Hat kaynagi: {PRED_FILE}")


In [ ]:
import json
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

for name in ["ROOT", "L1_DIR", "L2_DIR", "MODEL_DIR", "SPLIT_FILE", "FORCE_REBUILD_DATA"]:
    if name not in globals():
        raise RuntimeError(f"{name} tanimli degil — once kurulum hucresini kos.")

# ============================================================== ayarlar

LEVEL1_VARIANT = "lsp"   # hucre 3'teki VARIANT ile ayni olmali

N_Z_TOTAL = 512
N_Z_TRAIN = 358

EXPECTED_TRAIN_GROUPS = 89379
EXPECTED_VAL_GROUPS = 19122

L1_PRED = MODEL_DIR / f"level1_gbm_{LEVEL1_VARIANT}" / "predictions.csv"

LEGACY_TRAIN = L2_DIR / "train.csv"          # w_* kaynagi
LEGACY_VAL = L2_DIR / "validation.csv"

OUT_TRAIN = L2_DIR / f"dataset_train_{LEVEL1_VARIANT}.csv"
OUT_VAL = L2_DIR / f"dataset_validation_{LEVEL1_VARIANT}.csv"

# ============================================================== sutunlar

W_COLUMNS = ["WIdx_i11", "WIdx_i12", "WIdx_i2"]

PHYSICS_FEATURES = [
    "muSNR", "sigma2Wick", "wickCV2", "traceC", "Neff", "muPwrNorm",
    "diagEnergy", "offDiagEnRatio", "lambdaMaxNorm", "spectralEntropy",
    "meanTraceRatio",
]

AGG_COLUMNS = [f"{b}_{s}" for b in PHYSICS_FEATURES for s in ("meanZ", "stdZ")]

DIRECT_COLUMNS = [
    "w_traceUBR", "w_muBRPower", "w_nlosTraceUBR", "w_muBRShare", "w_nlosShare",
    "w_diagEnergyUBR", "w_diagPeakShareUBR", "w_diagCVUBR",
    "w_commonModeNormUBR", "w_polModeNormUBR", "w_modeBalanceUBR",
    "w_commonMeanModeNorm", "w_polMeanModeNorm",
    "w_s1Mean", "w_s1AbsMean", "w_s1PowerMean", "w_s1PeakShare", "w_s1CV",
    "w_s2Mean", "w_s2AbsMean", "w_s2PowerMean", "w_s2PeakShare", "w_s2CV",
    "w_s2ToS1PowerRatio", "w_s1s2Cosine",
]

SCALAR_COLUMNS = ["level1Hat", "level1StdHat"]

LABEL_COLUMNS = ["targetBankMean", "targetWMean", "targetDeltaW", "targetWStd", "nValidZ"]

OUTPUT_COLUMNS = (
    ["bankID", *W_COLUMNS, *SCALAR_COLUMNS, *AGG_COLUMNS, *DIRECT_COLUMNS,
     *LABEL_COLUMNS]
)

assert len(AGG_COLUMNS) == 22
assert len(DIRECT_COLUMNS) == 25

# ============================================================== onbellek

cached = OUT_TRAIN.exists() and OUT_VAL.exists() and not FORCE_REBUILD_DATA

if cached:
    level2_train = pd.read_csv(OUT_TRAIN)
    level2_val = pd.read_csv(OUT_VAL)

    for label, D in [("train", level2_train), ("validation", level2_val)]:
        missing = set(OUTPUT_COLUMNS) - set(D.columns)

        if missing:
            print(f"{label} eksik sutun, yeniden uretilecek: {sorted(missing)[:5]}")
            cached = False
            break

if cached:
    print("=" * 78)
    print("LEVEL 2 VERI SETI MEVCUT — yeniden uretilmedi")
    print("=" * 78)
    print(f"  train      : {len(level2_train):,} grup")
    print(f"  validation : {len(level2_val):,} grup")

else:
    t0 = time.time()

    print("=" * 78)
    print("LEVEL 2 VERI SETI KURULUYOR")
    print("=" * 78)

    if not L1_PRED.exists():
        raise RuntimeError(f"{L1_PRED} yok — once Level 1 GBM hucresini kos.")

    for p in [LEGACY_TRAIN, LEGACY_VAL]:
        if not p.exists():
            raise RuntimeError(
                f"{p} yok.\n"
                "25 w_* ozniteligi prepare_teacher_bank ile uretiliyor ve 3400 "
                "bank icin saatler suruyor. O adim bir kez kosulmus olmali."
            )

    # ---- z bolunmesi -----------------------------------------------------

    split = pd.read_parquet(SPLIT_FILE)

    pivot = (
        split.pivot(index="bankID", columns="zCandidate", values="isTrainZ")
        .sort_index().sort_index(axis=1).to_numpy(dtype=bool)
    )

    IS_TRAIN_Z = pivot[0]

    if not np.all(pivot == IS_TRAIN_Z[None, :]):
        raise RuntimeError("z bolunmesi global degil")

    if IS_TRAIN_Z.sum() != N_Z_TRAIN:
        raise RuntimeError("Egitim z sayisi uyusmuyor")

    del pivot, split

    print(f"z : {int(IS_TRAIN_Z.sum())} egitim / {int((~IS_TRAIN_Z).sum())} holdout")

    # ---- (bank, W) agregalari --------------------------------------------

    READ_COLUMNS = [
        "bankID", "zCandidate", *W_COLUMNS, "analyticValid",
        "targetLogVarRatio", *PHYSICS_FEATURES,
    ]

    def build_aggregates(split_name):
        files = sorted((ROOT / split_name).glob("*.parquet"))

        parts = []

        for i, path in enumerate(files, 1):
            D = pq.read_table(
                path, columns=READ_COLUMNS, use_threads=True
            ).to_pandas(split_blocks=True)

            D = D.loc[D["analyticValid"].to_numpy(dtype=bool)]

            if len(D) == 0:
                continue

            z = D["zCandidate"].to_numpy(np.int64)

            D = D.loc[IS_TRAIN_Z[z]]

            if len(D) == 0:
                continue

            keys = np.rec.fromarrays(
                [
                    D["bankID"].to_numpy(np.int64),
                    D["WIdx_i11"].to_numpy(np.int64),
                    D["WIdx_i12"].to_numpy(np.int64),
                    D["WIdx_i2"].to_numpy(np.int64),
                ],
                names=["bankID", *W_COLUMNS],
            )

            uniq, inverse = np.unique(keys, return_inverse=True)

            count = np.bincount(inverse).astype(np.float64)

            y = D["targetLogVarRatio"].to_numpy(np.float64)
            P = D[PHYSICS_FEATURES].to_numpy(np.float64)

            if not np.isfinite(y).all() or not np.isfinite(P).all():
                raise RuntimeError(f"{path.name}: NaN/Inf")

            y_mean = np.bincount(inverse, weights=y) / count
            y_var = np.bincount(inverse, weights=y * y) / count - y_mean**2

            part = pd.DataFrame(
                {
                    "bankID": uniq["bankID"].astype(np.int64),
                    "WIdx_i11": uniq["WIdx_i11"].astype(np.int64),
                    "WIdx_i12": uniq["WIdx_i12"].astype(np.int64),
                    "WIdx_i2": uniq["WIdx_i2"].astype(np.int64),
                    "targetWMean": y_mean,
                    "targetWStd": np.sqrt(np.maximum(y_var, 0.0)),
                    "nValidZ": count.astype(np.int64),
                }
            )

            for j, feature in enumerate(PHYSICS_FEATURES):
                x = P[:, j]

                m = np.bincount(inverse, weights=x) / count
                v = np.bincount(inverse, weights=x * x) / count - m**2

                part[f"{feature}_meanZ"] = m
                part[f"{feature}_stdZ"] = np.sqrt(np.maximum(v, 0.0))

            parts.append(part)

            if i % 60 == 0 or i == len(files):
                print(f"  [{split_name}] {i}/{len(files)} shard  [{time.time()-t0:.0f}s]")

        G = pd.concat(parts, ignore_index=True)

        # Ayni (bank,W) birden fazla shard'da olabilir — birlestir
        if G.duplicated(["bankID", *W_COLUMNS]).any():
            raise RuntimeError(f"{split_name}: (bank, W) anahtari tekrar ediyor")

        return G

    agg_train = build_aggregates("train")
    agg_val = build_aggregates("validation")

    print(f"\nAgrega: {len(agg_train):,} / {len(agg_val):,} grup")

    # ---- w_* oznitelikleri (mevcut dosyadan) ------------------------------

    def load_direct(path, label):
        D = pd.read_csv(path, usecols=["bankID", *W_COLUMNS, *DIRECT_COLUMNS])

        if D.duplicated(["bankID", *W_COLUMNS]).any():
            raise RuntimeError(f"{label}: w_* dosyasinda tekrar eden anahtar")

        return D

    dir_train = load_direct(LEGACY_TRAIN, "train")
    dir_val = load_direct(LEGACY_VAL, "validation")

    print(f"w_*   : {len(dir_train):,} / {len(dir_val):,} grup")

    # ---- Level 1 tahminleri ----------------------------------------------

    l1 = pd.read_csv(L1_PRED)

    needed = {"bankID", "hat_targetBankMean", "true_targetBankMean"}

    if not needed <= set(l1.columns):
        raise RuntimeError(f"{L1_PRED}: eksik sutun {sorted(needed - set(l1.columns))}")

    has_std = "hat_targetBankStd" in l1.columns

    l1_small = l1[["bankID", "hat_targetBankMean", "true_targetBankMean"]].copy()
    l1_small.columns = ["bankID", "level1Hat", "targetBankMean"]

    if has_std:
        l1_small["level1StdHat"] = l1["hat_targetBankStd"].to_numpy()
    else:
        print("UYARI: hat_targetBankStd yok, level1StdHat sifir olarak dolduruldu.")
        l1_small["level1StdHat"] = 0.0

    # ---- birlestir --------------------------------------------------------

    def assemble(agg, direct, label, expected):
        D = agg.merge(direct, on=["bankID", *W_COLUMNS], how="inner", validate="one_to_one")

        if len(D) != len(agg):
            raise RuntimeError(
                f"{label}: agrega {len(agg)} grup, w_* eslesmesi sonrasi {len(D)}"
            )

        D = D.merge(l1_small, on="bankID", how="left", validate="many_to_one")

        if D["level1Hat"].isna().any():
            n = int(D["level1Hat"].isna().sum())
            raise RuntimeError(f"{label}: {n} satirda Level 1 tahmini eslesmedi")

        D["targetDeltaW"] = D["targetWMean"] - D["targetBankMean"]

        if len(D) != expected:
            raise RuntimeError(f"{label}: {len(D)} grup, {expected} bekleniyor")

        return D[OUTPUT_COLUMNS].sort_values(["bankID", *W_COLUMNS]).reset_index(drop=True)

    level2_train = assemble(agg_train, dir_train, "train", EXPECTED_TRAIN_GROUPS)
    level2_val = assemble(agg_val, dir_val, "validation", EXPECTED_VAL_GROUPS)

    level2_train.to_csv(OUT_TRAIN, index=False)
    level2_val.to_csv(OUT_VAL, index=False)

    print()
    print(f"Yazildi [{(time.time()-t0)/60:.1f} dk]")
    print(f"  {OUT_TRAIN.name}")
    print(f"  {OUT_VAL.name}")

# ============================================================== dogrulama

print()
print("=" * 78)
print("DOGRULAMA")
print("=" * 78)

overlap = np.intersect1d(
    level2_train["bankID"].unique(), level2_val["bankID"].unique()
)

if overlap.size:
    raise RuntimeError(f"{overlap.size} bank hem egitimde hem dogrulamada")

for label, D in [("train", level2_train), ("validation", level2_val)]:
    resid = np.abs(
        D["targetDeltaW"] - (D["targetWMean"] - D["targetBankMean"])
    ).max()

    if resid > 1e-9:
        raise RuntimeError(f"{label}: targetDeltaW tutarsiz ({resid:.3e})")

    per_bank = D.groupby("bankID")["targetDeltaW"].mean().abs().max()

    l1_err = float(
        np.sqrt(np.mean((D["level1Hat"] - D["targetBankMean"]) ** 2))
    )

    print(f"{label:<11} grup={len(D):>7,}  bank={D['bankID'].nunique():>5,}  "
          f"deltaW bank-ici-ortalama |max|={per_bank:.2e}  "
          f"level1Hat RMSE={l1_err:.6f}")

print()
print("targetDeltaW bank icinde sifir ortalamali — bank_zero_loss'un")
print("dayandigi ozellik dogrulandi.")

# ---- level1Hat uyumu -------------------------------------------------------

tr_err = float(
    np.sqrt(np.mean((level2_train["level1Hat"] - level2_train["targetBankMean"]) ** 2))
)

va_err = float(
    np.sqrt(np.mean((level2_val["level1Hat"] - level2_val["targetBankMean"]) ** 2))
)

print()
print(f"level1Hat egitim/dogrulama orani : {tr_err / va_err:.3f}")
print("  1.0'a yakin olmali. Egitim tarafi out-of-fold uretilmeseydi ~0.3")
print("  olurdu ve Level 2 level1Hat'e hak ettiginden fazla guvenirdi.")

# ============================================================== sutun ozeti

print()
print("=" * 78)
print("GIRDI GRUPLARI")
print("=" * 78)
print(f"  scalars      : {len(SCALAR_COLUMNS):>3}   level1Hat, level1StdHat — baglamsiz")
print(f"  direct (w_*) : {len(DIRECT_COLUMNS):>3}   tam o (bank,W) cifti — baglamsiz")
print(f"  agg          : {len(AGG_COLUMNS):>3}   grubun 358 z'si gerekli — BAGLAMLI")
print(f"  etiket       : {len(LABEL_COLUMNS):>3}   ASLA GIRDI OLMAZ (targetWStd dahil)")
print()
print("Bank-goreli gorunumler egitim hucresinde uretilecek — onlar da")
print("bankin 32 W'sini gerektiriyor, yani BAGLAMLI.")

print()
print("HEDEF (targetDeltaW)")

for label, D in [("train", level2_train), ("validation", level2_val)]:
    y = D["targetDeltaW"]
    print(f"  {label:<11} mean={y.mean():+.3e}  std={y.std(ddof=0):.6f}  "
          f"[{y.min():+.3f}, {y.max():+.3f}]")

nz = level2_train["nValidZ"]

print()
print(f"Grup basina gecerli z : min {nz.min()}, medyan {int(nz.median())}, max {nz.max()}")

In [ ]:
"""
=============================================================================
HUCRE 5 — LEVEL 2 MLP
=============================================================================

HEDEF
  targetDeltaW = targetWMean - targetBankMean

  Level 1'in GERCEK hedefine gore artik. Cikti toplamsal:
      level2Hat = level1Hat + deltaHat

GIRDI GRUPLARI (INPUT_GROUPS ile acilip kapaniyor)
--------------------------------------------------
  env         94  ham ortam (78 sayisal + 16 senaryo one-hot)  BAGLAMSIZ
  scalars      2  level1Hat, level1StdHat                      BAGLAMSIZ
  w_emb        3  W indeksleri (embedding)                     BAGLAMSIZ
  direct_abs  25  w_* — tam o (bank,W) cifti icin              BAGLAMSIZ
  agg_abs     22  grubun 358 z'si uzerinden mean/std           BAGLAMLI
  direct_rel  50  w_*'in bank-goreli hali (merkez + std'ye bol) BAGLAMLI
  agg_rel     44  agregalarin bank-goreli hali                 BAGLAMLI

env, scalars ve w_emb her zaman acik. Digerleri INPUT_GROUPS listesinde.

ABLASYON PLANI
--------------
  tam          ["agg_abs", "direct_abs", "agg_rel", "direct_rel"]  referans
  mutlak       ["agg_abs", "direct_abs"]        bank-goreli gorunumlerin bedeli
  sadece W     ["direct_abs", "direct_rel"]     agregalarin bedeli
  baglamsiz    ["direct_abs"]                   tek (LSP,W,z) sorgusu icin
  fiziksiz     []                               fizigin toplam katkisi

Bu tablo iki soruyu birden cevapliyor: hangi grup ne tasiyor, ve tek
sorgulanabilir bir hattin bedeli ne.

MIMARI NOTU
-----------
Her grup KENDI kodlayicisina gidiyor ve 64 boyuta iniyor. Hepsini tek bir
kodlayicida birlestirmek denendi: delta RMSE 0.1007 yerine 0.1164 verdi.
Mutlak ve goreli gorunumlerin ayri islenmesi onemli — goreli gorunumler bu
seviyenin merkezinde.

KAYIP
-----
  MSE + 0.35 * (bank ici merkezlenmis MSE) + 0.10 * (bank ici ortalama)^2

Hedef bank icinde tanim geregi sifir ortalamali. Batch'ler TAM BANK
iceriyor, yoksa bu terimler hesaplanamaz.

ON KOSUL: Hucre 1-4.
=============================================================================
"""

import copy
import json
import time

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

for name in ["L1_DIR", "L2_DIR", "MODEL_DIR", "DEVICE", "FORCE_RETRAIN"]:
    if name not in globals():
        raise RuntimeError(f"{name} tanimli degil — once kurulum hucresini kos.")

# ============================================================== ayarlar

LEVEL1_VARIANT = "lsp"

INPUT_GROUPS = ["direct_abs"]
RUN_TAG = "deployment"     # ablasyon kosularini ayirmak icin

SEED = 42

BANKS_PER_BATCH = 64
MAX_EPOCHS = 200
PATIENCE = 25

LR = 5e-4
WEIGHT_DECAY = 2e-4
SCHED_PATIENCE = 7

LAMBDA_CENTER = 0.35
LAMBDA_ZERO = 0.10

LATENT_DIM = 96

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

VALID_GROUPS = ["agg_abs", "direct_abs", "agg_rel", "direct_rel", "direct_ref"]

for g in INPUT_GROUPS:
    if g not in VALID_GROUPS:
        raise RuntimeError(f"Bilinmeyen grup: {g}. Gecerli: {VALID_GROUPS}")

MODEL_ROOT = MODEL_DIR / f"level2_{LEVEL1_VARIANT}_{RUN_TAG}"
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

BEST_PT = MODEL_ROOT / "best.pt"

print("Device        :", DEVICE)
print("Level 1       :", LEVEL1_VARIANT)
print("Girdi gruplari:", INPUT_GROUPS if INPUT_GROUPS else "(fizik yok)")
print("Etiket        :", RUN_TAG)

t_start = time.time()

# ============================================================== veri

TRAIN_PATH = L2_DIR / f"dataset_train_{LEVEL1_VARIANT}.csv"
VAL_PATH = L2_DIR / f"dataset_validation_{LEVEL1_VARIANT}.csv"

for p in [TRAIN_PATH, VAL_PATH]:
    if not p.exists():
        raise RuntimeError(f"{p} yok — once Level 2 veri seti hucresini kos.")

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)

W_COLUMNS = ["WIdx_i11", "WIdx_i12", "WIdx_i2"]

PHYSICS_BASE = [
    "muSNR", "sigma2Wick", "wickCV2", "traceC", "Neff", "muPwrNorm",
    "diagEnergy", "offDiagEnRatio", "lambdaMaxNorm", "spectralEntropy",
    "meanTraceRatio",
]

AGG_COLUMNS = [f"{b}_{s}" for b in PHYSICS_BASE for s in ("meanZ", "stdZ")]

DIRECT_COLUMNS = [
    "w_traceUBR", "w_muBRPower", "w_nlosTraceUBR", "w_muBRShare", "w_nlosShare",
    "w_diagEnergyUBR", "w_diagPeakShareUBR", "w_diagCVUBR",
    "w_commonModeNormUBR", "w_polModeNormUBR", "w_modeBalanceUBR",
    "w_commonMeanModeNorm", "w_polMeanModeNorm",
    "w_s1Mean", "w_s1AbsMean", "w_s1PowerMean", "w_s1PeakShare", "w_s1CV",
    "w_s2Mean", "w_s2AbsMean", "w_s2PowerMean", "w_s2PeakShare", "w_s2CV",
    "w_s2ToS1PowerRatio", "w_s1s2Cosine",
]

BANK_train = train_df["bankID"].to_numpy(np.int64)
BANK_val = val_df["bankID"].to_numpy(np.int64)

Y_train = train_df["targetDeltaW"].to_numpy(np.float64)
Y_val = val_df["targetDeltaW"].to_numpy(np.float64)

L1_train = train_df["level1Hat"].to_numpy(np.float64)
L1_val = val_df["level1Hat"].to_numpy(np.float64)

WMEAN_val = val_df["targetWMean"].to_numpy(np.float64)

SCALARS_train = train_df[["level1Hat", "level1StdHat"]].to_numpy(np.float64)
SCALARS_val = val_df[["level1Hat", "level1StdHat"]].to_numpy(np.float64)

print()
print("=" * 78)
print("VERI")
print("=" * 78)
print(f"Grup      : {len(train_df):,} egitim / {len(val_df):,} dogrulama")
print(f"Bank      : {len(np.unique(BANK_train)):,} / {len(np.unique(BANK_val)):,}")
print(f"Hedef std : {Y_train.std():.6f}")

# ============================================================== ham ortam

l1_train = pd.read_csv(L1_DIR / "train.csv")
l1_val = pd.read_csv(L1_DIR / "validation.csv")

ENV_COLUMNS = ENV_FEATURES + DIRECTION_FEATURES

SCENARIO_COLUMNS = ["scenario_BR", "scenario_RU"]

l1_all = pd.concat([l1_train, l1_val], ignore_index=True)

LOG1P_RAW = {"K_BR", "K_RU", "dist_BR", "dist_RU"}


def transform_env(df):
    X = df[ENV_COLUMNS].to_numpy(np.float64, copy=True)

    for j, name in enumerate(ENV_COLUMNS):
        if name in LOG1P_RAW:
            X[:, j] = np.log1p(np.maximum(X[:, j], 0.0))

    if not np.isfinite(X).all():
        raise RuntimeError("Ortam donusumu NaN/Inf")

    return X


SCENARIOS = sorted(
    set(l1_train["scenario_BR"].astype(str)) | set(l1_train["scenario_RU"].astype(str))
)

SC_MAP = {s: i for i, s in enumerate(SCENARIOS)}
N_SC = len(SC_MAP)


def scenario_onehot(df):
    out = np.zeros((len(df), 2 * N_SC), dtype=np.float64)

    for k, col in enumerate(SCENARIO_COLUMNS):
        codes = df[col].astype(str).map(SC_MAP).to_numpy()
        ok = ~pd.isna(codes)
        out[np.nonzero(ok)[0], codes[ok].astype(np.int64) + k * N_SC] = 1.0

    return out


ENV_MATRIX = np.concatenate([transform_env(l1_all), scenario_onehot(l1_all)], axis=1)

env_by_bank = pd.DataFrame(ENV_MATRIX, index=l1_all["bankID"].to_numpy())

ENV_train = env_by_bank.reindex(BANK_train).to_numpy(np.float64)
ENV_val = env_by_bank.reindex(BANK_val).to_numpy(np.float64)

if np.isnan(ENV_train).any() or np.isnan(ENV_val).any():
    raise RuntimeError("Ortam oznitelikleri eslesmedi")

N_ENV = ENV_MATRIX.shape[1]

print(f"Ortam     : {N_ENV}  ({len(ENV_COLUMNS)} sayisal + {2*N_SC} one-hot)")

# ============================================================== fizik gruplari


def bank_relative(X, banks):
    """Bir ozniteligin bank ICINDEKI konumu: (merkezlenmis, std'ye bolunmus)."""
    _, inverse = np.unique(banks, return_inverse=True)

    n = inverse.max() + 1
    count = np.bincount(inverse).astype(np.float64)[:, None]

    mean = np.zeros((n, X.shape[1]))
    np.add.at(mean, inverse, X)
    mean /= count

    center = X - mean[inverse]

    var = np.zeros_like(mean)
    np.add.at(var, inverse, center**2)
    std = np.sqrt(var / count)

    return np.concatenate([center, center / (std[inverse] + 1e-6)], axis=1)


AGG_tr = train_df[AGG_COLUMNS].to_numpy(np.float64)
AGG_va = val_df[AGG_COLUMNS].to_numpy(np.float64)

DIR_tr = train_df[DIRECT_COLUMNS].to_numpy(np.float64)
DIR_va = val_df[DIRECT_COLUMNS].to_numpy(np.float64)

BLOCKS = {}

if "agg_abs" in INPUT_GROUPS:
    BLOCKS["agg_abs"] = (AGG_tr, AGG_va)

if "direct_abs" in INPUT_GROUPS:
    BLOCKS["direct_abs"] = (DIR_tr, DIR_va)

if "agg_rel" in INPUT_GROUPS:
    BLOCKS["agg_rel"] = (
        bank_relative(AGG_tr, BANK_train), bank_relative(AGG_va, BANK_val)
    )

if "direct_rel" in INPUT_GROUPS:
    BLOCKS["direct_rel"] = (
        bank_relative(DIR_tr, BANK_train), bank_relative(DIR_va, BANK_val)
    )

if "direct_ref" in INPUT_GROUPS:
    BLOCKS["direct_ref"] = (
        reference_block(train_df, DIR_tr),
        reference_block(val_df, DIR_va),
    )

BLOCK_NAMES = list(BLOCKS.keys())
BLOCK_DIMS = [BLOCKS[k][0].shape[1] for k in BLOCK_NAMES]

print()
print("FIZIK GRUPLARI")

if BLOCK_NAMES:
    for k, d in zip(BLOCK_NAMES, BLOCK_DIMS):
        ctx = "baglamli" if k.endswith("_rel") or k.startswith("agg") else "baglamsiz"
        print(f"  {k:<12}{d:>4}   {ctx}")
    print(f"  {'TOPLAM':<12}{sum(BLOCK_DIMS):>4}")
else:
    print("  (yok — fiziksiz ablasyon)")

W_train = train_df[W_COLUMNS].to_numpy(np.int64)
W_val = val_df[W_COLUMNS].to_numpy(np.int64)

W_VOCAB = [int(max(W_train[:, j].max(), W_val[:, j].max())) + 1 for j in range(3)]

print(f"\nW sozluk  : {W_VOCAB}")

# ============================================================== normalizasyon

_all_keys = pd.concat(
    [train_df[["bankID", *W_COLUMNS]], val_df[["bankID", *W_COLUMNS]]],
    ignore_index=True,
)

_cov = _all_keys.groupby(W_COLUMNS)["bankID"].nunique()
_n_banks = _all_keys["bankID"].nunique()

REF_W = tuple(int(v) for v in _cov.idxmax())
REF_COVERAGE = float(_cov.max() / _n_banks)

print(f"\nReferans W : i11={REF_W[0]}, i12={REF_W[1]}, i2={REF_W[2]}"
      f"   ({_cov.max()}/{_n_banks} bankta mevcut, %{100*REF_COVERAGE:.1f})")


def reference_block(df, direct_matrix):
    """Doner: (referansin mutlak degerleri, sorgunun referansa gore hali)."""
    mask = np.ones(len(df), dtype=bool)

    for col, v in zip(W_COLUMNS, REF_W):
        mask &= df[col].to_numpy() == v

    ref = (
        df.loc[mask, ["bankID", *DIRECT_COLUMNS]]
        .drop_duplicates("bankID")
        .set_index("bankID")
    )

    R = ref.reindex(df["bankID"].to_numpy()).to_numpy(np.float64)

    n_missing = int(np.isnan(R[:, 0]).sum())

    if n_missing:
        fallback = np.nanmedian(R, axis=0)
        R = np.where(np.isnan(R), fallback[None, :], R)
        print(f"  {n_missing} satirda referans W yok, medyan ile dolduruldu")

    # Kesin pozitif sutunlarda log-oran, digerlerinde fark
    pos = (direct_matrix > 0).all(axis=0) & (R > 0).all(axis=0)

    rel = np.where(
        pos,
        np.log(np.maximum(direct_matrix, 1e-30)) - np.log(np.maximum(R, 1e-30)),
        direct_matrix - R,
    )

    return np.concatenate([R, rel], axis=1)

def fit_norm(X):
    mu = X.mean(axis=0)
    sd = X.std(axis=0)
    sd[sd < 1e-8] = 1.0
    return mu, sd


def T(x):
    return torch.from_numpy(np.ascontiguousarray(x.astype(np.float32)))


env_mu, env_sd = fit_norm(ENV_train)
sc_mu, sc_sd = fit_norm(SCALARS_train)

TENV = {"train": T((ENV_train - env_mu) / env_sd), "val": T((ENV_val - env_mu) / env_sd)}
TSC = {
    "train": T((SCALARS_train - sc_mu) / sc_sd),
    "val": T((SCALARS_val - sc_mu) / sc_sd),
}

NORMS = {}
TBLOCK = {"train": [], "val": []}

for k in BLOCK_NAMES:
    tr, va = BLOCKS[k]
    mu, sd = fit_norm(tr)
    NORMS[k] = (mu, sd)

    TBLOCK["train"].append(T((tr - mu) / sd))
    TBLOCK["val"].append(T((va - mu) / sd))

TW = {"train": torch.from_numpy(W_train), "val": torch.from_numpy(W_val)}

TY_train = T(Y_train)

# ============================================================== model


class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout=0.08):
        super().__init__()

        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.LayerNorm(dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
        )

        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        return F.gelu(self.norm(x + self.block(x)))


class Level2PrecoderNet(nn.Module):
    """Her fizik grubu KENDI kodlayicisindan gecer. Hepsini tek kodlayicida
    birlestirmek denendi ve delta RMSE 0.1007 yerine 0.1164 verdi."""

    def __init__(self, n_env, n_scal, block_dims, w_vocab, latent_dim=LATENT_DIM):
        super().__init__()

        self.ctx_proj = nn.Linear(96, latent_dim)

        self.env_encoder = nn.Sequential(
            nn.Linear(n_env + n_scal, 160),
            nn.LayerNorm(160),
            nn.GELU(),
            nn.Dropout(0.05),
            nn.Linear(160, 96),
            nn.LayerNorm(96),
            nn.GELU(),
        )

        self.w_emb = nn.ModuleList(
            [nn.Embedding(v, d) for v, d in zip(w_vocab, (12, 12, 8))]
        )

        self.w_encoder = nn.Sequential(
            nn.Linear(32, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Linear(64, 32),
            nn.LayerNorm(32),
            nn.GELU(),
        )

        self.block_encoders = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Linear(d, max(96, min(160, 3 * d))),
                    nn.LayerNorm(max(96, min(160, 3 * d))),
                    nn.GELU(),
                    nn.Linear(max(96, min(160, 3 * d)), 64),
                    nn.LayerNorm(64),
                    nn.GELU(),
                )
                for d in block_dims
            ]
        )

        phys_out = 64 * len(block_dims)

        joint = 96 + 32 + phys_out

        self.candidate = nn.Sequential(
            nn.Linear(joint, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.05),
            nn.Linear(256, latent_dim),
        )

        self.gate = nn.Sequential(nn.Linear(joint, latent_dim), nn.Sigmoid())

        self.latent_norm = nn.LayerNorm(latent_dim)

        fusion_dim = 96 + latent_dim + 32 + phys_out + n_scal

        self.fusion = nn.Sequential(
            nn.Linear(fusion_dim, 320),
            nn.LayerNorm(320),
            nn.GELU(),
            nn.Dropout(0.08),
        )

        self.fusion_residual = ResidualBlock(320, dropout=0.08)

        self.head = nn.Sequential(
            nn.Linear(320, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.05),
            nn.Linear(128, 1),
        )

    def forward(self, env, scal, w_idx, blocks):
        ctx = self.env_encoder(torch.cat([env, scal], dim=1))

        w_raw = torch.cat([self.w_emb[j](w_idx[:, j]) for j in range(3)], dim=1)
        h_w = self.w_encoder(w_raw)

        if blocks:
            h_p = torch.cat(
                [enc(b) for enc, b in zip(self.block_encoders, blocks)], dim=1
            )
            joint = torch.cat([ctx, h_w, h_p], dim=1)
        else:
            h_p = ctx[:, :0]
            joint = torch.cat([ctx, h_w], dim=1)

        latent = self.latent_norm(
            self.ctx_proj(ctx) + self.gate(joint) * self.candidate(joint)
        )

        h = self.fusion(torch.cat([ctx, latent, h_w, h_p, scal], dim=1))

        h = self.fusion_residual(h)

        return self.head(h).squeeze(-1)


model = Level2PrecoderNet(N_ENV, SCALARS_train.shape[1], BLOCK_DIMS, W_VOCAB).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Parametre : {n_params:,}")

# ============================================================== batch

train_banks = np.unique(BANK_train)

bank_rows = {int(b): np.flatnonzero(BANK_train == b) for b in train_banks}


def gather(split, idx):
    return (
        TENV[split][idx].to(DEVICE),
        TSC[split][idx].to(DEVICE),
        TW[split][idx].to(DEVICE),
        [b[idx].to(DEVICE) for b in TBLOCK[split]],
    )


def compute_loss(pred, true, gidx, n_groups):
    base = F.mse_loss(pred, true)

    ones = torch.ones_like(pred)

    count = torch.zeros(n_groups, device=pred.device).index_add_(0, gidx, ones)

    p_mean = torch.zeros(n_groups, device=pred.device).index_add_(0, gidx, pred) / count
    t_mean = torch.zeros(n_groups, device=pred.device).index_add_(0, gidx, true) / count

    centered = F.mse_loss(pred - p_mean[gidx], true - t_mean[gidx])

    return base + LAMBDA_CENTER * centered + LAMBDA_ZERO * (p_mean**2).mean()


@torch.inference_mode()
def predict(split, n):
    model.eval()

    out = []

    for s in range(0, n, 8192):
        idx = torch.arange(s, min(s + 8192, n))
        out.append(model(*gather(split, idx)).float().cpu().numpy())

    return np.concatenate(out).astype(np.float64)


def stats(pred, true):
    err = pred - true
    return {
        "RMSE": float(np.sqrt(np.mean(err**2))),
        "MAE": float(np.mean(np.abs(err))),
        "R2": float(1 - np.sum(err**2) / np.sum((true - true.mean()) ** 2)),
        "bias": float(np.mean(err)),
    }


def evaluate():
    delta = predict("val", len(val_df))

    return stats(delta, Y_val), stats(L1_val + delta, WMEAN_val), delta


# ============================================================== egitim

cached = False

if BEST_PT.exists() and not FORCE_RETRAIN:
    ckpt = torch.load(BEST_PT, map_location=DEVICE, weights_only=False)

    if ckpt.get("blockNames") == BLOCK_NAMES and ckpt.get("nEnv") == N_ENV:
        model.load_state_dict(ckpt["state"])
        best_epoch = ckpt["bestEpoch"]
        history = pd.DataFrame(ckpt.get("history", []))
        cached = True

        print()
        print("MEVCUT MODEL KULLANILIYOR (en iyi epoch "
              f"{best_epoch}) — FORCE_RETRAIN = True ile yeniden egit")
    else:
        print("\nKaydedilmis model farkli girdilerle egitilmis, yeniden egitiliyor.")

if not cached:
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=SCHED_PATIENCE, min_lr=1e-6
    )

    best_rmse = np.inf
    best_state = None
    best_epoch = -1
    bad = 0

    history = []

    rng = np.random.default_rng(SEED)

    print()
    print("=" * 78)
    print("EGITIM")
    print("=" * 78)

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()

        order = rng.permutation(len(train_banks))

        loss_sum = 0.0
        n_steps = 0

        for s in range(0, len(order), BANKS_PER_BATCH):
            chunk = order[s : s + BANKS_PER_BATCH]

            if len(chunk) < 4:
                continue

            rows, gidx = [], []

            for k, bi in enumerate(chunk):
                r = bank_rows[int(train_banks[bi])]
                rows.append(r)
                gidx.append(np.full(len(r), k))

            idx = torch.from_numpy(np.concatenate(rows))
            g = torch.from_numpy(np.concatenate(gidx)).to(DEVICE)

            pred = model(*gather("train", idx))

            loss = compute_loss(pred, TY_train[idx].to(DEVICE), g, len(chunk))

            optimizer.zero_grad()
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)

            optimizer.step()

            loss_sum += float(loss.item())
            n_steps += 1

        d, l2, _ = evaluate()

        scheduler.step(d["RMSE"])

        history.append(
            {
                "epoch": epoch,
                "loss": loss_sum / max(n_steps, 1),
                "deltaRMSE": d["RMSE"],
                "deltaR2": d["R2"],
                "level2RMSE": l2["RMSE"],
                "level2R2": l2["R2"],
                "lr": optimizer.param_groups[0]["lr"],
            }
        )

        print(
            f"Epoch {epoch:03d} | loss={loss_sum/max(n_steps,1):.6f} | "
            f"delta={d['RMSE']:.6f} R2={d['R2']:.4f} | "
            f"L2={l2['RMSE']:.6f} R2={l2['R2']:.4f} | "
            f"bias={l2['bias']:+.2e} | lr={optimizer.param_groups[0]['lr']:.1e}"
        )

        if d["RMSE"] < best_rmse - 1e-6:
            best_rmse = d["RMSE"]
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            bad = 0
        else:
            bad += 1

        if bad >= PATIENCE:
            print(f"\nErken durdurma. En iyi epoch = {best_epoch}")
            break

    model.load_state_dict(best_state)

    history = pd.DataFrame(history)

# ============================================================== sonuc

d, l2, delta_val = evaluate()

baseline = float(np.sqrt(np.mean((L1_val - WMEAN_val) ** 2)))

print()
print("#" * 78)
print("EN IYI LEVEL 2 MODELI")
print("#" * 78)
print(f"Etiket        : {RUN_TAG}")
print(f"Girdi grubu   : {INPUT_GROUPS if INPUT_GROUPS else '(fizik yok)'}")
print(f"Fizik boyutu  : {sum(BLOCK_DIMS)}")
print(f"En iyi epoch  : {best_epoch}")

print("\nDELTA W")
for k in ["RMSE", "MAE", "R2"]:
    print(f"  {k:<5}: {d[k]:.6f}")
print(f"  bias : {d['bias']:+.3e}")

print("\nLEVEL 2 (level1Hat + deltaHat, targetWMean'e karsi)")
for k in ["RMSE", "MAE", "R2"]:
    print(f"  {k:<5}: {l2[k]:.6f}")
print(f"  bias : {l2['bias']:+.3e}")

print(f"\nBaseline (sadece level1Hat) : {baseline:.6f}")
print(f"Hata azalmasi               : %{100*(1-l2['RMSE']/baseline):.1f}")
print(f"Hedef std                   : {Y_train.std():.6f}")

l1_err = float(np.sqrt(np.mean((L1_val - (WMEAN_val - Y_val)) ** 2)))

print()
print("HATA BUTCESI")
print(f"  Level 1 : {l1_err:.6f}  -> %{100*l1_err**2/l2['RMSE']**2:.0f}")
print(f"  Delta   : {d['RMSE']:.6f}  -> %{100*d['RMSE']**2/l2['RMSE']**2:.0f}")

print()
print("ABLASYON TABLOSU (buraya her kosunun sonucunu ekle)")
print(f"  {RUN_TAG:<12} groups={INPUT_GROUPS}")
print(f"  {'':<12} delta RMSE={d['RMSE']:.6f}  L2 RMSE={l2['RMSE']:.6f}")

# ============================================================== kayit

torch.save(
    {
        "state": model.state_dict(),
        "bestEpoch": best_epoch,
        "runTag": RUN_TAG,
        "inputGroups": INPUT_GROUPS,
        "blockNames": BLOCK_NAMES,
        "blockDims": BLOCK_DIMS,
        "level1Variant": LEVEL1_VARIANT,
        "nEnv": N_ENV,
        "envColumns": ENV_COLUMNS,
        "scenarioMap": SC_MAP,
        "wVocab": W_VOCAB,
        "norm": {
            "envMu": env_mu, "envSd": env_sd,
            "scMu": sc_mu, "scSd": sc_sd,
            "blocks": {k: NORMS[k] for k in BLOCK_NAMES},
        },
        "metrics": {"delta": d, "level2": l2, "baseline": baseline},
        "history": history.to_dict("records") if len(history) else [],
    },
    BEST_PT,
)

if len(history):
    history.to_csv(MODEL_ROOT / "training_history.csv", index=False)

delta_train = predict("train", len(train_df))

for label, D, dh, l1 in [
    ("train", train_df, delta_train, L1_train),
    ("validation", val_df, delta_val, L1_val),
]:
    out = D[["bankID", *W_COLUMNS]].copy()

    out["level1Hat"] = l1
    out["level1StdHat"] = D["level1StdHat"].to_numpy()
    out["level2DeltaHat"] = dh
    out["level2Hat"] = l1 + dh

    out.to_csv(MODEL_ROOT / f"{label}_predictions.csv", index=False)

print()
print("Kaydedildi:", MODEL_ROOT)
print()
print("UYARI: Egitim tarafindaki level2DeltaHat, o gruplari GORMUS bir")
print("modelden geliyor (Level 1'de out-of-fold ile cozdugumuz sorunun")
print("aynisi). Level 3 bunu girdi olarak kullanacak; oradaki sonuclar")
print("okunurken akilda tutulmali.")

print(f"\nSure: {(time.time()-t_start)/60:.1f} dk")

In [ ]:
"""
=============================================================================
HUCRE 6 — MC'SIZ Z–KANAL ETKILESIM OZELLIKLERI
=============================================================================

Her (bank,W,z) icin 16 yeni ozellik yalniz deterministik kanal momentlerinden
hesaplanir. varEmpMC, targetLogVarRatio veya baska bir teacher/MC alani bu
hesaba girmez. Sonuclar kaynak teacher shard'lariyla ayni satir sirasinda,
ayri ve devam-edilebilir Parquet shard'larina yazilir.

Bu hucre uzun surebilir fakat her biten shard Drive'a atomik kaydedilir.
Runtime kesilirse hucreyi yeniden calistirmak yeterlidir.
=============================================================================
"""

import gc
import json
import os
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch

from ris_env.environment import build_deterministic_bank
from ris_env.label_engine import row_to_bank_input, build_w_candidate_pool
from ris_env.teacher_pipeline import enrich_environment_row_with_lsp
from ris_env.ris_response import generate_ris_response_from_z


from ris_env.interaction_features import (
    INTERACTION_FEATURE_NAMES, analytic_z_channel_interactions_multi_w,
)
from ris_env.label_engine import prepare_w_mean_only_state

@torch.inference_mode()
def _prepare_ubr(env, w):
    state = prepare_w_mean_only_state(env, w)
    return state.ubarBR, state.UBR

for name in [
    "ROOT", "ENV_CSV", "INTERACTION_DIR", "DEVICE",
    "FORCE_REBUILD_INTERACTIONS",
]:
    if name not in globals():
        raise RuntimeError(f"{name} tanimli degil — once kurulum hucresini kos.")

INTERACTION_VERSION = "z_channel_interactions_v1"
K_W = 32
N_Z = 512
LOCAL_INTERACTION_ROOT = Path("/content/z_channel_interactions_work")
LOCAL_INTERACTION_ROOT.mkdir(parents=True, exist_ok=True)

KEY_COLUMNS = [
    "bankID", "wCandidate", "zCandidate",
    "WIdx_i11", "WIdx_i12", "WIdx_i2",
]
FEATURE_COLUMNS = list(INTERACTION_FEATURE_NAMES)

env_df = pd.read_csv(ENV_CSV)
if env_df["bankID"].duplicated().any():
    raise RuntimeError("environments_4000.csv icinde tekrarli bankID var.")
env_by_bank = env_df.set_index("bankID", drop=False)

INTERACTION_DIR.mkdir(parents=True, exist_ok=True)

t0 = time.time()
done_banks = 0
done_rows = 0


def _bits_from_strings(strings, nris):
    strings = np.asarray(strings, dtype=str)
    if len(strings) != N_Z:
        raise RuntimeError(f"Z sayisi {len(strings)}, beklenen {N_Z}")
    if any(len(x) != nris for x in strings):
        raise RuntimeError("zString uzunlugu nRIS ile uyusmuyor.")
    bits = (
        np.frombuffer("".join(strings).encode("ascii"), dtype=np.uint8)
        .reshape(N_Z, nris)
        - ord("0")
    )
    if not np.isin(bits, [0, 1]).all():
        raise RuntimeError("zString 0/1 disinda deger iceriyor.")
    return bits


def _prepare_bank(row):
    # prepare_teacher_bank'in MC oncesi deterministik parcasi. Kaynak sharddaki
    # zString kullanildigi icin 512-Z havuzunu yeniden optimize etmiyoruz.
    raw = build_deterministic_bank(
        row_to_bank_input(row), device=DEVICE, parity=False
    )
    raw.pop("w_state", None)
    row_physical = enrich_environment_row_with_lsp(row, raw)
    W, WIdx = build_w_candidate_pool(
        row_physical, k_w=K_W, device=DEVICE, parity=False
    )
    return raw["static_env"], W, WIdx


def _valid_existing(out_path, source_rows):
    if not out_path.exists() or FORCE_REBUILD_INTERACTIONS:
        return False
    try:
        md = pq.read_metadata(out_path)
        names = set(pq.read_schema(out_path).names)
        return md.num_rows == source_rows and set(KEY_COLUMNS + FEATURE_COLUMNS) <= names
    except Exception:
        return False


print("=" * 78)
print("MC'SIZ Z–KANAL ETKILESIM OZELLIKLERI")
print("=" * 78)
print("Kaynak     :", ROOT)
print("Cikti      :", INTERACTION_DIR)
print("Ozellik    :", len(FEATURE_COLUMNS))
print("MC         : YOK")

for split_name in ("train", "validation"):
    source_files = sorted((ROOT / split_name).glob("*.parquet"))
    if not source_files:
        raise RuntimeError(f"{split_name} teacher shard'i bulunamadi.")

    split_out = INTERACTION_DIR / split_name
    split_out.mkdir(parents=True, exist_ok=True)
    local_split = LOCAL_INTERACTION_ROOT / split_name
    local_split.mkdir(parents=True, exist_ok=True)

    for shard_no, source_path in enumerate(source_files, 1):
        source_rows = pq.read_metadata(source_path).num_rows
        out_path = split_out / source_path.name

        if _valid_existing(out_path, source_rows):
            print(f"[{split_name} {shard_no:03d}/{len(source_files):03d}] mevcut: {out_path.name}")
            done_rows += source_rows
            continue

        available = set(pq.read_schema(source_path).names)
        required = set(KEY_COLUMNS + ["zString"])
        missing = required - available
        if missing:
            raise RuntimeError(f"{source_path.name}: eksik sutun {sorted(missing)}")

        D = pq.read_table(
            source_path, columns=KEY_COLUMNS + ["zString"], use_threads=True
        ).to_pandas(split_blocks=True)

        F = np.full((len(D), len(FEATURE_COLUMNS)), np.nan, dtype=np.float32)
        bank_values = D["bankID"].to_numpy(np.int64)

        for bank_id in pd.unique(bank_values):
            pos = np.flatnonzero(bank_values == int(bank_id))
            B = D.iloc[pos]

            if int(bank_id) not in env_by_bank.index:
                raise RuntimeError(f"bankID={bank_id}: environment satiri yok.")
            env_row = env_by_bank.loc[int(bank_id)]

            zref = (
                B[["zCandidate", "zString"]]
                .drop_duplicates()
                .sort_values("zCandidate")
            )
            if len(zref) != N_Z or not np.array_equal(
                zref["zCandidate"].to_numpy(np.int64), np.arange(N_Z)
            ):
                raise RuntimeError(f"bankID={bank_id}: 0..511 Z kontrati bozuk.")
            if B.groupby("zCandidate")["zString"].nunique().max() != 1:
                raise RuntimeError(f"bankID={bank_id}: W'ler arasinda zString degisiyor.")

            nris_x = int(env_row["nRIS1"])
            nris_y = int(env_row["nRIS2"])
            nris = 2 * nris_x * nris_y
            z = _bits_from_strings(zref["zString"].to_numpy(), nris)

            static_env, W, WIdx = _prepare_bank(env_row)
            got_widx = WIdx.detach().cpu().numpy().astype(np.int64)
            src_w = (
                B[["wCandidate", "WIdx_i11", "WIdx_i12", "WIdx_i2"]]
                .drop_duplicates()
                .sort_values("wCandidate")
            )
            if len(src_w) != K_W or not np.array_equal(
                src_w["wCandidate"].to_numpy(np.int64), np.arange(K_W)
            ):
                raise RuntimeError(f"bankID={bank_id}: 32-W kontrati bozuk.")
            if not np.array_equal(
                src_w[["WIdx_i11", "WIdx_i12", "WIdx_i2"]].to_numpy(np.int64),
                got_widx,
            ):
                raise RuntimeError(
                    f"bankID={bank_id}: mevcut shard ile yeniden kurulan WIdx uyusmuyor."
                )

            feat = analytic_z_channel_interactions_multi_w(
                static_env,
                W,
                torch.from_numpy(z).to(DEVICE),
                nris_x=nris_x,
                nris_y=nris_y,
                parity=False,
            ).detach().cpu().numpy().astype(np.float32)

            wc = B["wCandidate"].to_numpy(np.int64)
            zc = B["zCandidate"].to_numpy(np.int64)
            F[pos] = feat[wc, zc]

            done_banks += 1
            if done_banks % 10 == 0:
                print(
                    f"  bank={done_banks:4d}/3400  "
                    f"sure={(time.time()-t0)/60:.1f} dk"
                )

            del feat, static_env, W, WIdx
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        if not np.isfinite(F).all():
            raise RuntimeError(f"{source_path.name}: interaction NaN/Inf veya eksik satir.")

        out = D[KEY_COLUMNS].copy()
        for j, name in enumerate(FEATURE_COLUMNS):
            out[name] = F[:, j]

        local_tmp = local_split / (source_path.name + ".partial")
        local_final = local_split / source_path.name
        local_tmp.unlink(missing_ok=True)
        local_final.unlink(missing_ok=True)

        pq.write_table(
            pa.Table.from_pandas(out, preserve_index=False),
            local_tmp,
            compression="zstd",
        )
        os.replace(local_tmp, local_final)

        copying = out_path.with_suffix(".parquet.copying")
        copying.unlink(missing_ok=True)
        shutil.copy2(local_final, copying)
        os.replace(copying, out_path)

        if pq.read_metadata(out_path).num_rows != len(D):
            raise RuntimeError(f"{out_path.name}: kayit sonrasi satir sayisi bozuk.")

        done_rows += len(D)
        print(
            f"[{split_name} {shard_no:03d}/{len(source_files):03d}] "
            f"PASS rows={len(D):,}  {out_path.name}"
        )

manifest = {
    "version": INTERACTION_VERSION,
    "source": ROOT.name,
    "environment": ENV_CSV.name,
    "featureNames": FEATURE_COLUMNS,
    "featureCount": len(FEATURE_COLUMNS),
    "usesMonteCarlo": False,
    "usesTeacherTarget": False,
    "keyColumns": KEY_COLUMNS,
    "completedRows": int(done_rows),
    "updatedUnix": time.time(),
}
with open(INTERACTION_DIR / "manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print()
print("ETKILESIM OZELLIKLERI HAZIR")
print(f"Satir : {done_rows:,}")
print(f"Sure  : {(time.time()-t0)/60:.1f} dk")
print("MC    : calistirilmadi")


In [ ]:
"""
=============================================================================
HUCRE 7 — MC'SIZ KANAL HARITALARI
=============================================================================

Her (bank,W) grubu icin RIS yuzeyindeki bes normalize haritayi cikarir:
  1) BS->RIS toplam uyartim, 2) RIS->UE toplam kazanc,
  3) uctan uca kosegen onem, 4-5) iki hopun LOS paylari.

Hedef/MC alani okunmaz. Haritalar Z'den bagimsiz oldugu icin her grup icin
yalniz bir kez saklanir. Level 3, bu haritalari gamma(z) ile model icinde
faz-duyarli olarak birlestirir.

Her biten shard Drive'a atomik kaydedilir; kesilirse kaldigi yerden devam eder.
=============================================================================
"""

import gc
import json
import os
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch

CHANNEL_MAP_FEATURE_NAMES = (
    "map_log_br_excitation",
    "map_log_ru_gain",
    "map_log_e2e_diag",
    "map_br_los_fraction",
    "map_ru_los_fraction",
)

if "channel_map" not in Z_INPUTS:
    print("Kanal haritasi kapali; cikarim atlandi.")
else:
    for name in [
        "ROOT", "ENV_CSV", "HT_DIR", "DEVICE", "FORCE_REBUILD_CHANNEL_MAPS",
        "env_by_bank", "_prepare_bank", "_prepare_ubr", "K_W",
    ]:
        if name not in globals():
            raise RuntimeError(f"{name} tanimli degil — onceki hucreleri kos.")

    CHANNEL_MAP_VERSION = "z_channel_maps_v1"
    CHANNEL_MAP_FEATURE_NAMES = (
        "map_log_br_excitation",
        "map_log_ru_gain",
        "map_log_e2e_diag",
        "map_br_los_fraction",
        "map_ru_los_fraction",
    )
    CHANNEL_MAP_KEY_COLUMNS = ["bankID", "WIdx_i11", "WIdx_i12", "WIdx_i2"]

    CHANNEL_MAP_DIR = HT_DIR / CHANNEL_MAP_VERSION
    LOCAL_CHANNEL_MAP_ROOT = Path("/content/z_channel_maps_work")
    CHANNEL_MAP_DIR.mkdir(parents=True, exist_ok=True)
    LOCAL_CHANNEL_MAP_ROOT.mkdir(parents=True, exist_ok=True)

    MAX_NRIS = int(env_by_bank["nRIS"].max())
    MAP_EPS = 1e-12


    def _log_mean_profile(x):
        x = torch.clamp(x.float(), min=0.0)
        scale = torch.clamp(x.mean(), min=MAP_EPS)
        return torch.log1p(x / scale)


    @torch.inference_mode()
    def _channel_maps_multi_w(static_env, W):
        """Donus: [K_W, 5, nRIS], float32; tum guc haritalari olceksiz."""
        nris = int(static_env.muBR.shape[0])
        nr = int(static_env.muRU.shape[0])

        r_los = torch.sum(torch.abs(static_env.muRU).square(), dim=0).float()
        r_nlos = torch.clamp(
            nr * torch.diagonal(static_env.rho_RUhop).real.float(), min=0.0
        )
        r_total = r_los + r_nlos
        r_log = _log_mean_profile(r_total)
        r_los_frac = torch.where(
            r_total > MAP_EPS,
            r_los / torch.clamp(r_total, min=MAP_EPS),
            torch.zeros_like(r_total),
        ).clamp(0.0, 1.0)

        out = torch.empty(
            (len(W), len(CHANNEL_MAP_FEATURE_NAMES), nris),
            dtype=torch.float32,
            device=static_env.muBR.device,
        )

        for wi, w in enumerate(W):
            ubar, ubr = _prepare_ubr(static_env, w)
            u_total = torch.clamp(torch.diagonal(ubr).real.float(), min=0.0)
            u_los = torch.abs(ubar).square().float()
            d_total = u_total * r_total

            u_los_frac = torch.where(
                u_total > MAP_EPS,
                u_los / torch.clamp(u_total, min=MAP_EPS),
                torch.zeros_like(u_total),
            ).clamp(0.0, 1.0)

            out[wi] = torch.stack(
                [
                    _log_mean_profile(u_total),
                    r_log,
                    _log_mean_profile(d_total),
                    u_los_frac,
                    r_los_frac,
                ],
                dim=0,
            )

        if not torch.isfinite(out).all():
            raise RuntimeError("Kanal haritasinda NaN/Inf bulundu.")
        return out


    def _valid_map_file(path):
        if not path.exists() or FORCE_REBUILD_CHANNEL_MAPS:
            return False
        try:
            with np.load(path, allow_pickle=False) as Z:
                return (
                    str(Z["version"].item()) == CHANNEL_MAP_VERSION
                    and list(Z["feature_names"].astype(str))
                    == list(CHANNEL_MAP_FEATURE_NAMES)
                    and Z["keys"].ndim == 2
                    and Z["keys"].shape[1] == len(CHANNEL_MAP_KEY_COLUMNS)
                    and Z["maps"].shape
                    == (len(Z["keys"]), len(CHANNEL_MAP_FEATURE_NAMES), MAX_NRIS)
                    and Z["shapes"].shape == (len(Z["keys"]), 2)
                )
        except Exception:
            return False


    t0 = time.time()
    done_banks = 0
    done_groups = 0

    print("=" * 78)
    print("MC'SIZ KANAL HARITALARI")
    print("=" * 78)
    print("Kaynak     :", ROOT)
    print("Cikti      :", CHANNEL_MAP_DIR)
    print("Harita     :", len(CHANNEL_MAP_FEATURE_NAMES))
    print("Maks nRIS  :", MAX_NRIS)
    print("MC         : YOK")

    for split_name in ("train", "validation"):
        source_files = sorted((ROOT / split_name).glob("*.parquet"))
        if not source_files:
            raise RuntimeError(f"{split_name} teacher shard'i bulunamadi.")

        split_out = CHANNEL_MAP_DIR / split_name
        local_split = LOCAL_CHANNEL_MAP_ROOT / split_name
        split_out.mkdir(parents=True, exist_ok=True)
        local_split.mkdir(parents=True, exist_ok=True)

        for shard_no, source_path in enumerate(source_files, 1):
            out_path = split_out / f"{source_path.stem}.npz"
            if _valid_map_file(out_path):
                with np.load(out_path, allow_pickle=False) as Z:
                    done_groups += len(Z["keys"])
                print(
                    f"[{split_name} {shard_no:03d}/{len(source_files):03d}] "
                    f"mevcut: {out_path.name}"
                )
                continue

            required = {"bankID", "wCandidate", "WIdx_i11", "WIdx_i12", "WIdx_i2"}
            available = set(pq.read_schema(source_path).names)
            missing = required - available
            if missing:
                raise RuntimeError(f"{source_path.name}: eksik sutun {sorted(missing)}")

            D = pq.read_table(
                source_path,
                columns=["bankID", "wCandidate", "WIdx_i11", "WIdx_i12", "WIdx_i2"],
                use_threads=True,
            ).to_pandas(split_blocks=True).drop_duplicates()

            all_keys, all_shapes, all_maps = [], [], []

            for bank_id in pd.unique(D["bankID"]):
                if int(bank_id) not in env_by_bank.index:
                    raise RuntimeError(f"bankID={bank_id}: environment satiri yok.")

                env_row = env_by_bank.loc[int(bank_id)]
                nris_x = int(env_row["nRIS1"])
                nris_y = int(env_row["nRIS2"])
                nris = 2 * nris_x * nris_y
                if nris != int(env_row["nRIS"]):
                    raise RuntimeError(f"bankID={bank_id}: nRIS boyut kontrati bozuk.")

                B = D.loc[D["bankID"] == bank_id].sort_values("wCandidate")
                if len(B) != K_W or not np.array_equal(
                    B["wCandidate"].to_numpy(np.int64), np.arange(K_W)
                ):
                    raise RuntimeError(f"bankID={bank_id}: 32-W kontrati bozuk.")

                static_env, W, WIdx = _prepare_bank(env_row)
                got_widx = WIdx.detach().cpu().numpy().astype(np.int64)
                src_widx = B[["WIdx_i11", "WIdx_i12", "WIdx_i2"]].to_numpy(np.int64)
                if not np.array_equal(src_widx, got_widx):
                    raise RuntimeError(
                        f"bankID={bank_id}: kaynak ile yeniden kurulan WIdx uyusmuyor."
                    )

                maps = _channel_maps_multi_w(static_env, W).detach().cpu().numpy()
                padded = np.zeros(
                    (K_W, len(CHANNEL_MAP_FEATURE_NAMES), MAX_NRIS), dtype=np.float16
                )
                padded[:, :, :nris] = maps.astype(np.float16)

                keys = np.column_stack(
                    [np.full(K_W, int(bank_id), dtype=np.int64), got_widx]
                )
                shapes = np.tile(
                    np.asarray([nris_x, nris_y], dtype=np.int16), (K_W, 1)
                )

                all_keys.append(keys)
                all_shapes.append(shapes)
                all_maps.append(padded)

                done_banks += 1
                if done_banks % 10 == 0:
                    print(
                        f"  bank={done_banks:4d}/3400  "
                        f"sure={(time.time()-t0)/60:.1f} dk"
                    )

                del maps, padded, static_env, W, WIdx
                gc.collect()

            keys = np.concatenate(all_keys, axis=0)
            shapes = np.concatenate(all_shapes, axis=0)
            maps = np.concatenate(all_maps, axis=0)
            if not np.isfinite(maps).all():
                raise RuntimeError(f"{source_path.name}: kanal haritasinda NaN/Inf.")

            local_tmp = local_split / f"{source_path.stem}.partial.npz"
            local_final = local_split / f"{source_path.stem}.npz"
            local_tmp.unlink(missing_ok=True)
            local_final.unlink(missing_ok=True)

            with open(local_tmp, "wb") as f:
                np.savez_compressed(
                    f,
                    version=np.asarray(CHANNEL_MAP_VERSION),
                    feature_names=np.asarray(CHANNEL_MAP_FEATURE_NAMES),
                    key_columns=np.asarray(CHANNEL_MAP_KEY_COLUMNS),
                    keys=keys,
                    shapes=shapes,
                    maps=maps,
                )
            os.replace(local_tmp, local_final)

            copying = out_path.with_suffix(".npz.copying")
            copying.unlink(missing_ok=True)
            shutil.copy2(local_final, copying)
            os.replace(copying, out_path)

            if not _valid_map_file(out_path):
                raise RuntimeError(f"{out_path.name}: kayit sonrasi kontrol basarisiz.")

            done_groups += len(keys)
            print(
                f"[{split_name} {shard_no:03d}/{len(source_files):03d}] "
                f"PASS grup={len(keys):,}  {out_path.name}"
            )

    manifest = {
        "version": CHANNEL_MAP_VERSION,
        "source": ROOT.name,
        "environment": ENV_CSV.name,
        "featureNames": list(CHANNEL_MAP_FEATURE_NAMES),
        "featureCount": len(CHANNEL_MAP_FEATURE_NAMES),
        "keyColumns": CHANNEL_MAP_KEY_COLUMNS,
        "maxNRIS": MAX_NRIS,
        "dtype": "float16",
        "usesMonteCarlo": False,
        "usesTeacherTarget": False,
        "completedGroups": int(done_groups),
        "updatedUnix": time.time(),
    }
    with open(CHANNEL_MAP_DIR / "manifest.json", "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    print()
    print("KANAL HARITALARI HAZIR")
    print(f"Grup  : {done_groups:,}")
    print(f"Sure  : {(time.time()-t0)/60:.1f} dk")
    print("MC    : calistirilmadi")



In [ ]:
"""Istege bagli tam kovaryans; sadece Z_INPUTS icinde covariance varsa calisir.
RU/BR sirasiyla LL,LN,NL,NN paylari ve normalize off-diagonal katkilari.
Teacher hedefleri okunmaz; tamamlanan shard'lar tekrar kullanilir.
"""
@torch.inference_mode()
def covariance_multi_w(static_env, W, z, *, nris_x, nris_y, parity=False):
    gamma = generate_ris_response_from_z(z, device=static_env.muBR.device,
                                         parity=parity)["gamma"]
    parts = []
    for w in W:
        ubar, ubr = _prepare_ubr(static_env, w)
        parts.append(covariance_component_features(static_env, ubr, ubar, gamma))
    return torch.stack(parts, dim=0)

def build_covariance_sidecars():
    for name in [
        "ROOT", "ENV_CSV", "COVARIANCE_DIR", "DEVICE",
        "FORCE_REBUILD_COVARIANCE",
    ]:
        if name not in globals():
            raise RuntimeError(f"{name} tanimli degil — once kurulum hucresini kos.")

    COVARIANCE_VERSION = "z_covariance_v1"
    K_W = 32
    N_Z = 512
    LOCAL_COVARIANCE_ROOT = Path("/content/z_covariance_work")
    LOCAL_COVARIANCE_ROOT.mkdir(parents=True, exist_ok=True)

    KEY_COLUMNS = [
        "bankID", "wCandidate", "zCandidate",
        "WIdx_i11", "WIdx_i12", "WIdx_i2",
    ]
    FEATURE_COLUMNS = list(COVARIANCE_FEATURE_NAMES)

    env_df = pd.read_csv(ENV_CSV)
    if env_df["bankID"].duplicated().any():
        raise RuntimeError("environments_4000.csv icinde tekrarli bankID var.")
    env_by_bank = env_df.set_index("bankID", drop=False)

    COVARIANCE_DIR.mkdir(parents=True, exist_ok=True)

    t0 = time.time()
    done_banks = 0
    done_rows = 0


    def _bits_from_strings(strings, nris):
        strings = np.asarray(strings, dtype=str)
        if len(strings) != N_Z:
            raise RuntimeError(f"Z sayisi {len(strings)}, beklenen {N_Z}")
        if any(len(x) != nris for x in strings):
            raise RuntimeError("zString uzunlugu nRIS ile uyusmuyor.")
        bits = (
            np.frombuffer("".join(strings).encode("ascii"), dtype=np.uint8)
            .reshape(N_Z, nris)
            - ord("0")
        )
        if not np.isin(bits, [0, 1]).all():
            raise RuntimeError("zString 0/1 disinda deger iceriyor.")
        return bits


    def _prepare_bank(row):
        # prepare_teacher_bank'in MC oncesi deterministik parcasi. Kaynak sharddaki
        # zString kullanildigi icin 512-Z havuzunu yeniden optimize etmiyoruz.
        raw = build_deterministic_bank(
            row_to_bank_input(row), device=DEVICE, parity=False
        )
        raw.pop("w_state", None)
        row_physical = enrich_environment_row_with_lsp(row, raw)
        W, WIdx = build_w_candidate_pool(
            row_physical, k_w=K_W, device=DEVICE, parity=False
        )
        return raw["static_env"], W, WIdx


    import hashlib
    def source_digest(path):
        h = hashlib.sha256()
        with open(path, "rb") as stream:
            for block in iter(lambda: stream.read(4 * 1024 * 1024), b""):
                h.update(block)
        return h.hexdigest()
    env_digest = source_digest(ENV_CSV)

    def _valid_existing(out_path, source_rows, source_hash):
        if not out_path.exists() or FORCE_REBUILD_COVARIANCE:
            return False
        try:
            md = pq.read_metadata(out_path)
            names = set(pq.read_schema(out_path).names)
            stored = md.metadata or {}
            return (md.num_rows == source_rows and set(KEY_COLUMNS + FEATURE_COLUMNS) <= names
                    and stored.get(b"covarianceVersion") == COVARIANCE_VERSION.encode()
                    and stored.get(b"sourceSHA256") == source_hash.encode()
                    and stored.get(b"environmentSHA256") == env_digest.encode())
        except Exception:
            return False


    print("=" * 78)
    print("MC'SIZ Z–KANAL KOVARYANS OZELLIKLERI")
    print("=" * 78)
    print("Kaynak     :", ROOT)
    print("Cikti      :", COVARIANCE_DIR)
    print("Ozellik    :", len(FEATURE_COLUMNS))
    print("MC         : YOK")

    for split_name in ("train", "validation"):
        source_files = sorted((ROOT / split_name).glob("*.parquet"))
        if not source_files:
            raise RuntimeError(f"{split_name} teacher shard'i bulunamadi.")

        split_out = COVARIANCE_DIR / split_name
        split_out.mkdir(parents=True, exist_ok=True)
        local_split = LOCAL_COVARIANCE_ROOT / split_name
        local_split.mkdir(parents=True, exist_ok=True)

        for shard_no, source_path in enumerate(source_files, 1):
            source_rows = pq.read_metadata(source_path).num_rows
            out_path = split_out / source_path.name

            source_hash = source_digest(source_path)
            if _valid_existing(out_path, source_rows, source_hash):
                print(f"[{split_name} {shard_no:03d}/{len(source_files):03d}] mevcut: {out_path.name}")
                done_rows += source_rows
                continue

            available = set(pq.read_schema(source_path).names)
            required = set(KEY_COLUMNS + ["zString"])
            missing = required - available
            if missing:
                raise RuntimeError(f"{source_path.name}: eksik sutun {sorted(missing)}")

            D = pq.read_table(
                source_path, columns=KEY_COLUMNS + ["zString"], use_threads=True
            ).to_pandas(split_blocks=True)

            F = np.full((len(D), len(FEATURE_COLUMNS)), np.nan, dtype=np.float32)
            bank_values = D["bankID"].to_numpy(np.int64)

            for bank_id in pd.unique(bank_values):
                pos = np.flatnonzero(bank_values == int(bank_id))
                B = D.iloc[pos]

                if int(bank_id) not in env_by_bank.index:
                    raise RuntimeError(f"bankID={bank_id}: environment satiri yok.")
                env_row = env_by_bank.loc[int(bank_id)]

                zref = (
                    B[["zCandidate", "zString"]]
                    .drop_duplicates()
                    .sort_values("zCandidate")
                )
                if len(zref) != N_Z or not np.array_equal(
                    zref["zCandidate"].to_numpy(np.int64), np.arange(N_Z)
                ):
                    raise RuntimeError(f"bankID={bank_id}: 0..511 Z kontrati bozuk.")
                if B.groupby("zCandidate")["zString"].nunique().max() != 1:
                    raise RuntimeError(f"bankID={bank_id}: W'ler arasinda zString degisiyor.")

                nris_x = int(env_row["nRIS1"])
                nris_y = int(env_row["nRIS2"])
                nris = 2 * nris_x * nris_y
                z = _bits_from_strings(zref["zString"].to_numpy(), nris)

                static_env, W, WIdx = _prepare_bank(env_row)
                got_widx = WIdx.detach().cpu().numpy().astype(np.int64)
                src_w = (
                    B[["wCandidate", "WIdx_i11", "WIdx_i12", "WIdx_i2"]]
                    .drop_duplicates()
                    .sort_values("wCandidate")
                )
                if len(src_w) != K_W or not np.array_equal(
                    src_w["wCandidate"].to_numpy(np.int64), np.arange(K_W)
                ):
                    raise RuntimeError(f"bankID={bank_id}: 32-W kontrati bozuk.")
                if not np.array_equal(
                    src_w[["WIdx_i11", "WIdx_i12", "WIdx_i2"]].to_numpy(np.int64),
                    got_widx,
                ):
                    raise RuntimeError(
                        f"bankID={bank_id}: mevcut shard ile yeniden kurulan WIdx uyusmuyor."
                    )

                feat = covariance_multi_w(
                    static_env,
                    W,
                    torch.from_numpy(z).to(DEVICE),
                    nris_x=nris_x,
                    nris_y=nris_y,
                    parity=False,
                ).detach().cpu().numpy().astype(np.float32)

                wc = B["wCandidate"].to_numpy(np.int64)
                zc = B["zCandidate"].to_numpy(np.int64)
                F[pos] = feat[wc, zc]

                done_banks += 1
                if done_banks % 10 == 0:
                    print(
                        f"  bank={done_banks:4d}/3400  "
                        f"sure={(time.time()-t0)/60:.1f} dk"
                    )

                del feat, static_env, W, WIdx
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            if not np.isfinite(F).all():
                raise RuntimeError(f"{source_path.name}: covariance NaN/Inf veya eksik satir.")

            out = D[KEY_COLUMNS].copy()
            for j, name in enumerate(FEATURE_COLUMNS):
                out[name] = F[:, j]

            local_tmp = local_split / (source_path.name + ".partial")
            local_final = local_split / source_path.name
            local_tmp.unlink(missing_ok=True)
            local_final.unlink(missing_ok=True)

            pq.write_table(
                pa.Table.from_pandas(out, preserve_index=False).replace_schema_metadata({
                    b"covarianceVersion": COVARIANCE_VERSION.encode(),
                    b"sourceSHA256": source_hash.encode(),
                    b"environmentSHA256": env_digest.encode(),
                }),
                local_tmp,
                compression="zstd",
            )
            os.replace(local_tmp, local_final)

            copying = out_path.with_suffix(".parquet.copying")
            copying.unlink(missing_ok=True)
            shutil.copy2(local_final, copying)
            os.replace(copying, out_path)

            if pq.read_metadata(out_path).num_rows != len(D):
                raise RuntimeError(f"{out_path.name}: kayit sonrasi satir sayisi bozuk.")

            done_rows += len(D)
            print(
                f"[{split_name} {shard_no:03d}/{len(source_files):03d}] "
                f"PASS rows={len(D):,}  {out_path.name}"
            )

    manifest = {
        "version": COVARIANCE_VERSION,
        "source": ROOT.name,
        "environment": ENV_CSV.name,
        "environmentSHA256": env_digest,
        "featureNames": FEATURE_COLUMNS,
        "featureCount": len(FEATURE_COLUMNS),
        "usesMonteCarlo": False,
        "usesTeacherTarget": False,
        "keyColumns": KEY_COLUMNS,
        "completedRows": int(done_rows),
        "updatedUnix": time.time(),
    }
    temporary = COVARIANCE_DIR / "manifest.json.partial"
    temporary.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    os.replace(temporary, COVARIANCE_DIR / "manifest.json")

    print()
    print("KOVARYANS OZELLIKLERI HAZIR")
    print(f"Satir : {done_rows:,}")
    print(f"Sure  : {(time.time()-t0)/60:.1f} dk")
    print("MC    : calistirilmadi")

if "covariance" in Z_INPUTS:
    build_covariance_sidecars()
else:
    print("Kovaryans kapali; yeni sidecar hesaplanmadi.")


In [ ]:
"""
=============================================================================
HUCRE 8 — LEVEL 3 VERI SETI
=============================================================================

HEDEF
    targetDeltaZ = targetLogVarRatio(bank, W, z) - targetWMean(bank, W)

targetWMean, o (bank, W) grubunun 358 EGITIM z'si uzerinden alinan GERCEK
ortalama. Level 2'nin targetDeltaW tanimiyla simetrik: model temiz bir
hedef ogrenir, ust kademelerin hatasi degerlendirmede durustce ustune biner.

2x2 DEGERLENDIRME IZGARASI
--------------------------
Iki bagimsiz genelleme ekseni var ve simdiye kadar sadece birini olcmustuk:

                    egitim z (358)      holdout z (154)
  egitim bank       EGITIM VERISI       yeni desen, bilinen geometri
  dogrulama bank    yeni geometri       ikisi de yeni

Bu yuzden kurucu TUM 512 z'yi TUM 3400 bank icin yaziyor ve her satira
isTrainZ + isTrainBank bayraklarini koyuyor. Filtreleme egitim hucresinde.

SATIR SUTUNLARI
---------------
  groupIdx      group_table.parquet'teki satir indeksi
  zCandidate    0..511
  10 fizik      muSNR, sigma2Wick, wickCV2, traceC, Neff, diagEnergy,
                offDiagEnRatio, lambdaMaxNorm, spectralEntropy,
                meanTraceRatio   — hepsi ANALITIK, MC turevli degil
  16 interaction  z faz durumunun W-kosullu BR uyartimi ve RIS-tarafi RU
                  profiliyle etkilesimi — ANALITIK, MC turevli degil
  targetDeltaZ  hedef
  isTrainZ      z bolunmesi (global, tum banklarda ayni)
  isTrainBank   bank bolunmesi

muPwrNorm satir seviyesinde SABIT cikmisti (z ile degismiyor), o yuzden
10 metrik var, 11 degil.

BEYAZ LISTE
-----------
Parquet'te 114 sutun ve cogu Monte Carlo turevli. targetLogVarRatio tanim
geregi log(varRatio); varRatio, cv2, meanEmpMC, varEmpMC, muSNRLegacy,
q05GG, logQ05GG, ggShapeA hepsi MC'den geliyor. N_MC_used ve
mcFinalP90Delta gibi yakinsama teshisleri de disarida — gercek kullanimda
MC kosmadan onlara sahip olamazsin. Bu hucre yalnizca acikca listelenen
sutunlari okur.

z DESENLERI
-----------
zCandidate INDEKSI kullanilamaz: bolunme global oldugu icin holdout'taki
154 indeks egitimde hic gorunmez; bir nn.Embedding(512, d) o vektorleri
hic guncellemez. Bunun yerine desenin KENDISI saklaniyor.

Desenler BANK'A OZGU — 512 adayin sadece 68'i banklar arasi ortak
(64 structured + 4 anchor). RIS boyutu da 4 farkli deger aliyor
(8x4, 8x8, 16x8, 16x16), o yuzden bit sekli bank basina degisiyor.

Bit sirasi (nRIS1, nRIS2, 2) olarak ampirik dogrulandi: structured
desenlerin komsu-hucre puruzlulugu rastgele desenlerin 0.38'i.

CIKTILAR
--------
  level3/rows/{train,validation}_*.parquet
  level3/group_table.parquet
  level3/bank_table.parquet
  level3/z_patterns.npy            paketlenmis bitler (nBank, 512, maxByte)
  level3/z_pattern_features.npy    (nBank, 512, 13) float32
  level3/z_nearest_dist.npy        (nBank, 512) int16  — kopya tespiti
  level3/build_manifest.json

Etkilesim ozellikleri onceki hucrede z_channel_interactions_v1 altinda
kaydedilir ve burada teacher satirlariyla anahtar/sira kontrolunden sonra
birlestirilir.

ASAMALI: her asama kendi ciktisini kontrol eder, yarida kesilirse kaldigi
yerden devam eder.

ON KOSUL: Hucre 1-5.
=============================================================================
"""

import json
import time

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

for name in ["ROOT", "L1_DIR", "L2_DIR", "L3_DIR", "MODEL_DIR", "SPLIT_FILE",
             "INTERACTION_DIR", "INTERACTION_FEATURE_NAMES", "FORCE_REBUILD_DATA"]:
    if name not in globals():
        raise RuntimeError(f"{name} tanimli degil — once kurulum hucresini kos.")

# ============================================================== ayarlar

LEVEL1_VARIANT = "lsp"
LEVEL2_TAG = "deployment"        # REF'siz varyant

N_Z_TOTAL = 512
N_Z_TRAIN = 358

CLEAN_DIST_RATIO = 0.04          # nRIS'in %4'unden yakin desen = kopya

ROW_PHYSICS = [
    "muSNR", "sigma2Wick", "wickCV2", "traceC", "Neff",
    "diagEnergy", "offDiagEnRatio", "lambdaMaxNorm", "spectralEntropy",
    "meanTraceRatio",
]

INTERACTION_FEATURES = list(INTERACTION_FEATURE_NAMES)
INTERACTION_KEY_COLUMNS = [
    "bankID", "wCandidate", "zCandidate",
    "WIdx_i11", "WIdx_i12", "WIdx_i2",
]

interaction_manifest_path = INTERACTION_DIR / "manifest.json"
if not interaction_manifest_path.exists():
    raise RuntimeError(
        f"{interaction_manifest_path} yok — once MC'siz etkilesim hucresini kos."
    )
with open(interaction_manifest_path, encoding="utf-8") as f:
    interaction_manifest = json.load(f)
if interaction_manifest.get("featureNames") != INTERACTION_FEATURES:
    raise RuntimeError("Etkilesim manifesti ile kodun ozellik listesi uyusmuyor.")

FORBIDDEN = {
    "varRatio", "cv2", "meanEmpMC", "varEmpMC", "muSNRLegacy",
    "q05GG", "logQ05GG", "ggShapeA", "N_MC_used", "N_MC_base",
    "mcRefined", "mcConverged", "mcEarlyP99", "mcFinalP90Delta",
    "lookupClamped", "channel_seed", "ris_seed", "ch_seed",
    "targetWStd",
}

assert not (set(ROW_PHYSICS) & FORBIDDEN)
assert not (set(INTERACTION_FEATURES) & FORBIDDEN)

W_COLUMNS = ["WIdx_i11", "WIdx_i12", "WIdx_i2"]

PATTERN_FEATURE_NAMES = [
    "density", "densityPol0", "densityPol1", "polBalance", "polAgreement",
    "rowStd", "colStd", "roughRow", "roughCol", "transitionRate",
    "fftLow", "fftPeak", "quadBalance",
]

ROWS_DIR = L3_DIR / "rows"
ROWS_DIR.mkdir(parents=True, exist_ok=True)

L2_MODEL = MODEL_DIR / f"level2_{LEVEL1_VARIANT}_{LEVEL2_TAG}"

META_FILES = [
    L3_DIR / "group_table.parquet",
    L3_DIR / "bank_table.parquet",
    L3_DIR / "z_patterns.npy",
    L3_DIR / "z_pattern_features.npy",
    L3_DIR / "z_nearest_dist.npy",
]

t0 = time.time()

old_build_manifest = L3_DIR / "build_manifest.json"
REBUILD_ROW_SHARDS = FORCE_REBUILD_DATA
if old_build_manifest.exists() and not FORCE_REBUILD_DATA:
    with open(old_build_manifest, encoding="utf-8") as f:
        old_build = json.load(f)
    if old_build.get("interactionFeatures") != INTERACTION_FEATURES:
        REBUILD_ROW_SHARDS = True
        print("Eski Level 3 satirlari etkilesim ozelligi icermiyor; yeniden yazilacak.")
elif any(ROWS_DIR.glob("*.parquet")) and not FORCE_REBUILD_DATA:
    REBUILD_ROW_SHARDS = True
    print("Tamamlanmis Level 3 manifesti yok; satir shard'lari yeniden yazilacak.")


def elapsed():
    return f"{time.time() - t0:.0f}s"


print("=" * 78)
print("LEVEL 3 VERI SETI")
print("=" * 78)
print(f"Level 1 : {LEVEL1_VARIANT}")
print(f"Level 2 : {LEVEL2_TAG}   ({L2_MODEL.name})")

# ============================================================== z bolunmesi

split = pd.read_parquet(SPLIT_FILE)

pivot = (
    split.pivot(index="bankID", columns="zCandidate", values="isTrainZ")
    .sort_index().sort_index(axis=1).to_numpy(dtype=bool)
)

IS_TRAIN_Z = pivot[0]

if not np.all(pivot == IS_TRAIN_Z[None, :]):
    raise RuntimeError("z bolunmesi global degil")

if IS_TRAIN_Z.sum() != N_Z_TRAIN:
    raise RuntimeError("Egitim z sayisi uyusmuyor")

del pivot, split

TRAIN_Z_IDX = np.nonzero(IS_TRAIN_Z)[0]
HOLD_Z_IDX = np.nonzero(~IS_TRAIN_Z)[0]

print(f"z       : {len(TRAIN_Z_IDX)} egitim / {len(HOLD_Z_IDX)} holdout (global)")

# ============================================================== ASAMA A

meta_done = all(p.exists() for p in META_FILES) and not FORCE_REBUILD_DATA

if meta_done:
    print("\nASAMA A atlandi (meta veri mevcut).")

    GROUPS = pd.read_parquet(L3_DIR / "group_table.parquet")
    BANKS = pd.read_parquet(L3_DIR / "bank_table.parquet")

else:
    print()
    print("-" * 78)
    print("ASAMA A — META VERI")
    print("-" * 78)

    # ---- grup tablosu: Level 2 ciktilariyla -----------------------------

    l2_train = pd.read_csv(L2_DIR / f"dataset_train_{LEVEL1_VARIANT}.csv")
    l2_val = pd.read_csv(L2_DIR / f"dataset_validation_{LEVEL1_VARIANT}.csv")

    KEEP = ["bankID", *W_COLUMNS, "level1Hat", "level1StdHat",
            "targetBankMean", "targetWMean"]

    l2_train = l2_train[KEEP].copy()
    l2_val = l2_val[KEEP].copy()

    l2_train["isTrainBank"] = True
    l2_val["isTrainBank"] = False

    GROUPS = pd.concat([l2_train, l2_val], ignore_index=True)

    # Level 2'nin delta tahminleri
    for label, path in [
        ("train", L2_MODEL / "train_predictions.csv"),
        ("validation", L2_MODEL / "validation_predictions.csv"),
    ]:
        if not path.exists():
            raise RuntimeError(
                f"{path} yok — once Level 2 MLP hucresini "
                f"RUN_TAG='{LEVEL2_TAG}' ile kos."
            )

    l2p = pd.concat(
        [
            pd.read_csv(L2_MODEL / "train_predictions.csv"),
            pd.read_csv(L2_MODEL / "validation_predictions.csv"),
        ],
        ignore_index=True,
    )[["bankID", *W_COLUMNS, "level2DeltaHat", "level2Hat"]]

    before = len(GROUPS)

    GROUPS = GROUPS.merge(
        l2p, on=["bankID", *W_COLUMNS], how="left", validate="one_to_one"
    )

    assert len(GROUPS) == before

    if GROUPS["level2DeltaHat"].isna().any():
        n = int(GROUPS["level2DeltaHat"].isna().sum())
        raise RuntimeError(f"{n} grupta level2DeltaHat eslesmedi")

    GROUPS = GROUPS.sort_values(["isTrainBank", "bankID", *W_COLUMNS],
                                ascending=[False, True, True, True, True])

    GROUPS = GROUPS.reset_index(drop=True)
    GROUPS["groupIdx"] = np.arange(len(GROUPS), dtype=np.int32)

    n_train_g = int(GROUPS["isTrainBank"].sum())

    # egitim gruplari bitisik blok olmali
    assert GROUPS["isTrainBank"].to_numpy()[:n_train_g].all()
    assert not GROUPS["isTrainBank"].to_numpy()[n_train_g:].any()

    print(f"Grup    : {n_train_g:,} egitim bank / "
          f"{len(GROUPS)-n_train_g:,} dogrulama bank")

    # ---- bank tablosu: geometri + desenler ------------------------------

    print(f"\nGeometri ve z desenleri okunuyor... [{elapsed()}]")

    def collect(split_name, is_train):
        files = sorted((ROOT / split_name).glob("*.parquet"))

        geo, patterns, type_ref = [], {}, None

        for i, path in enumerate(files, 1):
            D = pq.read_table(
                path,
                columns=["bankID", "zCandidate", "zString", "candidateType",
                         "nRIS1", "nRIS2", "nRIS"],
            ).to_pandas()

            geo.append(
                D.drop_duplicates("bankID")[["bankID", "nRIS1", "nRIS2", "nRIS"]]
            )

            Z = D.drop_duplicates(["bankID", "zCandidate"])

            for bank, sub in Z.groupby("bankID", sort=False):
                sub = sub.sort_values("zCandidate")

                if len(sub) != N_Z_TOTAL:
                    raise RuntimeError(f"bank {bank}: {len(sub)} z adayi")

                patterns[int(bank)] = sub["zString"].to_numpy()

                if type_ref is None:
                    type_ref = sub["candidateType"].to_numpy()

            if i % 60 == 0 or i == len(files):
                print(f"  {split_name} {i}/{len(files)} [{elapsed()}]")

        G = (
            pd.concat(geo, ignore_index=True).drop_duplicates("bankID")
            .sort_values("bankID").reset_index(drop=True)
        )

        G["isTrainBank"] = is_train

        return G, patterns, type_ref

    geo_tr, pat_tr, type_ref = collect("train", True)
    geo_va, pat_va, type_ref_va = collect("validation", False)

    if not np.array_equal(type_ref, type_ref_va):
        raise RuntimeError("candidateType duzeni splitler arasinda farkli")

    BANKS = pd.concat([geo_tr, geo_va], ignore_index=True)
    BANKS = BANKS.sort_values(["isTrainBank", "bankID"],
                              ascending=[False, True]).reset_index(drop=True)

    BANKS["bankRow"] = np.arange(len(BANKS), dtype=np.int32)

    bad = BANKS["nRIS"] != 2 * BANKS["nRIS1"] * BANKS["nRIS2"]

    if bad.any():
        raise RuntimeError(f"{int(bad.sum())} bankta nRIS != 2*nRIS1*nRIS2")

    print(f"\nBank    : {len(BANKS):,}")
    print(
        BANKS.groupby(["nRIS1", "nRIS2", "nRIS"]).size()
        .reset_index(name="bank").to_string(index=False)
    )

    ALL_PAT = {**pat_tr, **pat_va}

    del pat_tr, pat_va

    # ---- paketle, oznitelik cikar, kopya olc ---------------------------

    print(f"\nDesenler isleniyor... [{elapsed()}]")

    bank_ids = BANKS["bankID"].to_numpy(np.int64)
    n_banks = len(bank_ids)

    MAX_BITS = int(BANKS["nRIS"].max())
    MAX_BYTES = MAX_BITS // 8

    PATTERNS = np.zeros((n_banks, N_Z_TOTAL, MAX_BYTES), dtype=np.uint8)
    FEATURES = np.zeros((n_banks, N_Z_TOTAL, len(PATTERN_FEATURE_NAMES)), np.float32)
    NEAREST = np.zeros((n_banks, N_Z_TOTAL), dtype=np.int16)

    POPCOUNT = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)

    for row, bank in enumerate(bank_ids):
        n1 = int(BANKS.at[row, "nRIS1"])
        n2 = int(BANKS.at[row, "nRIS2"])
        nris = int(BANKS.at[row, "nRIS"])

        strings = ALL_PAT[int(bank)]

        bits = (
            np.frombuffer("".join(strings).encode(), dtype=np.uint8)
            .reshape(N_Z_TOTAL, nris) - ord("0")
        )

        packed = np.packbits(bits, axis=1)

        PATTERNS[row, :, : packed.shape[1]] = packed

        FEATURES[row] = physical_pattern_features(bits, n1, n2)

        # her aday icin en yakin EGITIM deseni (kendisi haric)
        d = POPCOUNT[
            np.bitwise_xor(packed[:, None, :], packed[None, TRAIN_Z_IDX, :])
        ].sum(axis=2, dtype=np.int16)

        for k, zi in enumerate(TRAIN_Z_IDX):
            d[zi, k] = nris

        NEAREST[row] = d.min(axis=1)

        if (row + 1) % 500 == 0:
            print(f"  {row+1}/{n_banks} bank [{elapsed()}]")

    del ALL_PAT

    BANKS["candidateTypeRef"] = ""   # tip listesi ayrica saklaniyor

    dirty = NEAREST / BANKS["nRIS"].to_numpy(np.float64)[:, None] < CLEAN_DIST_RATIO

    hold_dirty = dirty[:, HOLD_Z_IDX]

    print()
    print("KOPYA ANALIZI (holdout z, esik = nRIS'in %"
          f"{100*CLEAN_DIST_RATIO:.0f}'i)")
    print(f"  bank basina kirli : min {hold_dirty.sum(1).min()}, "
          f"medyan {int(np.median(hold_dirty.sum(1)))}, "
          f"max {hold_dirty.sum(1).max()}")

    tdf = pd.DataFrame(
        {
            "type": np.tile(type_ref[HOLD_Z_IDX], n_banks),
            "dirty": hold_dirty.ravel(),
        }
    )

    print("  tipe gore kirli orani (%):")
    print((100 * tdf.groupby("type")["dirty"].mean()).round(1).to_string())

    # ---- kaydet ---------------------------------------------------------

    GROUPS.to_parquet(L3_DIR / "group_table.parquet", index=False)
    BANKS.to_parquet(L3_DIR / "bank_table.parquet", index=False)

    np.save(L3_DIR / "z_patterns.npy", PATTERNS)
    np.save(L3_DIR / "z_pattern_features.npy", FEATURES)
    np.save(L3_DIR / "z_nearest_dist.npy", NEAREST)

    pd.DataFrame(
        {"zCandidate": np.arange(N_Z_TOTAL), "candidateType": type_ref,
         "isTrainZ": IS_TRAIN_Z}
    ).to_parquet(L3_DIR / "z_index_meta.parquet", index=False)

    print(f"\nMeta veri kaydedildi [{elapsed()}]")
    print(f"  desenler     : {PATTERNS.nbytes/1e6:.0f} MB")
    print(f"  oznitelikler : {FEATURES.nbytes/1e6:.0f} MB")

    del PATTERNS, FEATURES, NEAREST

# Fiziksel desen ozellikleri ayri dosyada; eski baseline korunur.
from ris_env.l3_features import physical_pattern_features, PATTERN_VERSION
import hashlib
import os

PATTERN_FILE_V2 = L3_DIR / "z_pattern_features_physical_v2.npy"
PATTERN_MANIFEST_V2 = L3_DIR / "z_pattern_features_physical_v2.json"

def _sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(4 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

pattern_signature = {
    "version": PATTERN_VERSION,
    "packedSHA256": _sha256_file(L3_DIR / "z_patterns.npy"),
    "bankSHA256": _sha256_file(L3_DIR / "bank_table.parquet"),
    "featureNames": PATTERN_FEATURE_NAMES,
}
pattern_current = False
if PATTERN_MANIFEST_V2.exists() and PATTERN_FILE_V2.exists() and not FORCE_REBUILD_PATTERNS:
    with open(PATTERN_MANIFEST_V2, encoding="utf-8") as stream:
        pattern_current = json.load(stream) == pattern_signature
    if pattern_current:
        check = np.load(PATTERN_FILE_V2, mmap_mode="r")
        pattern_current = check.shape == (len(BANKS), N_Z_TOTAL, len(PATTERN_FEATURE_NAMES))
        del check
if not pattern_current:
    packed = np.load(L3_DIR / "z_patterns.npy", mmap_mode="r")
    fixed = np.empty((len(BANKS), N_Z_TOTAL, len(PATTERN_FEATURE_NAMES)), np.float32)
    for _, row in BANKS.iterrows():
        idx, n1, n2 = int(row.bankRow), int(row.nRIS1), int(row.nRIS2)
        nris = 2 * n1 * n2
        bits = np.unpackbits(packed[idx], axis=-1, count=nris)
        fixed[idx] = physical_pattern_features(bits, n1, n2)
    if not np.isfinite(fixed).all():
        raise RuntimeError("Fiziksel desen ozelliklerinde NaN/Inf")
    partial = PATTERN_FILE_V2.with_suffix(".npy.partial")
    with open(partial, "wb") as stream:
        np.save(stream, fixed)
    os.replace(partial, PATTERN_FILE_V2)
    tmp_manifest = PATTERN_MANIFEST_V2.with_suffix(".json.partial")
    tmp_manifest.write_text(json.dumps(pattern_signature, indent=2), encoding="utf-8")
    os.replace(tmp_manifest, PATTERN_MANIFEST_V2)
    del fixed, packed
    print("Fiziksel desen ozellikleri yenilendi; eski L3 satirlari korundu.")
else:
    print("Fiziksel desen ozellikleri mevcut: v2")

# ============================================================== ASAMA B

print()
print("-" * 78)
print("ASAMA B — SATIR SHARD'LARI")
print("-" * 78)
print("TUM 512 z, TUM banklar yaziliyor (2x2 izgara icin).")

group_key = pd.MultiIndex.from_arrays(
    [GROUPS["bankID"], GROUPS["WIdx_i11"], GROUPS["WIdx_i12"], GROUPS["WIdx_i2"]]
)

GROUP_LOOKUP = pd.Series(GROUPS["groupIdx"].to_numpy(), index=group_key)

if not GROUP_LOOKUP.index.is_unique:
    raise RuntimeError("Grup anahtari benzersiz degil")

TARGET_W_MEAN = GROUPS["targetWMean"].to_numpy(np.float64)
IS_TRAIN_BANK = GROUPS["isTrainBank"].to_numpy(dtype=bool)

READ_COLUMNS = (
    ["bankID", "wCandidate", "zCandidate", *W_COLUMNS,
     "analyticValid", "targetLogVarRatio"]
    + ROW_PHYSICS
)

SCHEMA = pa.schema(
    [pa.field("groupIdx", pa.int32()), pa.field("zCandidate", pa.int16())]
    + [pa.field(c, pa.float32()) for c in ROW_PHYSICS]
    + [pa.field(c, pa.float32()) for c in INTERACTION_FEATURES]
    + [
        pa.field("targetDeltaZ", pa.float32()),
        pa.field("isTrainZ", pa.bool_()),
        pa.field("isTrainBank", pa.bool_()),
    ]
)


def process(split_name):
    files = sorted((ROOT / split_name).glob("*.parquet"))

    written = skipped = dropped = 0

    for i, path in enumerate(files, 1):
        out_path = ROWS_DIR / f"{split_name}_{i:04d}.parquet"

        if out_path.exists() and not REBUILD_ROW_SHARDS:
            skipped += 1
            continue

        D = pq.read_table(path, columns=READ_COLUMNS, use_threads=True).to_pandas(
            split_blocks=True
        )

        interaction_path = INTERACTION_DIR / split_name / path.name
        if not interaction_path.exists():
            raise RuntimeError(
                f"{interaction_path} yok — once MC'siz etkilesim hucresini tamamla."
            )
        I = pq.read_table(
            interaction_path,
            columns=INTERACTION_KEY_COLUMNS + INTERACTION_FEATURES,
            use_threads=True,
        ).to_pandas(split_blocks=True)
        if len(I) != len(D):
            raise RuntimeError(f"{path.name}: teacher/interaction satir sayisi farkli.")
        for key_col in INTERACTION_KEY_COLUMNS:
            if not np.array_equal(D[key_col].to_numpy(), I[key_col].to_numpy()):
                raise RuntimeError(
                    f"{path.name}: teacher/interaction satir sirasi {key_col} alaninda bozuk."
                )
        for feature in INTERACTION_FEATURES:
            D[feature] = I[feature].to_numpy(np.float32)
        del I

        D = D.loc[D["analyticValid"].to_numpy(dtype=bool)]

        if len(D) == 0:
            continue

        key = pd.MultiIndex.from_arrays(
            [D["bankID"], D["WIdx_i11"], D["WIdx_i12"], D["WIdx_i2"]]
        )

        gidx = GROUP_LOOKUP.reindex(key).to_numpy()

        keep = ~pd.isna(gidx)
        dropped += int((~keep).sum())

        if not keep.any():
            continue

        D = D.loc[keep]
        gidx = gidx[keep].astype(np.int32)

        z = D["zCandidate"].to_numpy(np.int64)

        if z.min() < 0 or z.max() >= N_Z_TOTAL:
            raise RuntimeError("zCandidate araligi bozuk")

        y = D["targetLogVarRatio"].to_numpy(np.float64)
        P = D[ROW_PHYSICS].to_numpy(np.float64)
        X = D[INTERACTION_FEATURES].to_numpy(np.float64)

        if (
            not np.isfinite(y).all()
            or not np.isfinite(P).all()
            or not np.isfinite(X).all()
        ):
            raise RuntimeError(f"{path.name}: NaN/Inf")

        arrays = [
            pa.array(gidx, type=pa.int32()),
            pa.array(z.astype(np.int16), type=pa.int16()),
        ]

        for j in range(len(ROW_PHYSICS)):
            arrays.append(pa.array(P[:, j].astype(np.float32), type=pa.float32()))

        for j in range(len(INTERACTION_FEATURES)):
            arrays.append(pa.array(X[:, j].astype(np.float32), type=pa.float32()))

        arrays.append(
            pa.array((y - TARGET_W_MEAN[gidx]).astype(np.float32), type=pa.float32())
        )
        arrays.append(pa.array(IS_TRAIN_Z[z], type=pa.bool_()))
        arrays.append(pa.array(IS_TRAIN_BANK[gidx], type=pa.bool_()))

        pq.write_table(
            pa.Table.from_arrays(arrays, schema=SCHEMA), out_path, compression="zstd"
        )

        written += 1

        if written % 40 == 0:
            print(f"  [{split_name}] {i}/{len(files)}, {written} yazildi [{elapsed()}]")

    print(f"  [{split_name}] {written} yazildi, {skipped} atlandi, "
          f"{dropped:,} satir grup bulamadi")


process("train")
process("validation")

# ============================================================== manifest

shards = sorted(ROWS_DIR.glob("*.parquet"))

n_rows = sum(pq.read_metadata(p).num_rows for p in shards)
out_bytes = sum(p.stat().st_size for p in shards)

manifest = {
    "level1Variant": LEVEL1_VARIANT,
    "level2Tag": LEVEL2_TAG,
    "source": ROOT.name,
    "rowPhysics": ROW_PHYSICS,
    "interactionFeatures": INTERACTION_FEATURES,
    "interactionVersion": interaction_manifest.get("version"),
    "forbiddenColumns": sorted(FORBIDDEN),
    "patternFeatures": PATTERN_FEATURE_NAMES,
    "target": "targetLogVarRatio - targetWMean",
    "nZTotal": N_Z_TOTAL,
    "nZTrain": N_Z_TRAIN,
    "cleanDistRatio": CLEAN_DIST_RATIO,
    "nGroups": int(len(GROUPS)),
    "nBanks": int(len(BANKS)),
    "totalRows": int(n_rows),
    "shardCount": len(shards),
    "outputBytes": int(out_bytes),
}

with open(L3_DIR / "build_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print()
print("=" * 78)
print("TAMAMLANDI")
print("=" * 78)
print(f"Sure   : {(time.time()-t0)/60:.1f} dk")
print(f"Satir  : {n_rows:,}")
print(f"Shard  : {len(shards)}")
print(f"Disk   : {out_bytes/1e9:.2f} GB")

expected = len(GROUPS) * N_Z_TOTAL

print()
print(f"Beklenen satir : {expected:,}  ({len(GROUPS):,} grup x {N_Z_TOTAL} z)")
print(f"Gercek         : {n_rows:,}")

if n_rows != expected:
    print(f"Fark {expected - n_rows:,} — analyticValid=False olan satirlar.")

print()
print("2x2 IZGARA (egitim hucresi bu bayraklarla filtreleyecek)")
print(f"  egitim bank + egitim z    : egitim verisi")
print(f"  egitim bank + holdout z   : yeni desen, bilinen geometri")
print(f"  dogrulama bank + egitim z : yeni geometri, bilinen desen")
print(f"  dogrulama bank + holdout z: ikisi de yeni")


In [ ]:
"""
=============================================================================
HUCRE 9 — LEVEL 3 SINIR AGI
=============================================================================

HEDEF
    targetDeltaZ = targetLogVarRatio(bank, W, z) - targetWMean(bank, W)

Hedefin std'si 0.1389 (Level 2'nin hedefi 0.3267'ydi — sinyal yari yariya
kucuk).

z TEMSILI (Z_INPUTS ile acilip kapaniyor)
-----------------------------------------
  row_phys  10  o (bank,W,z) uclusu icin analitik metrikler  1 degerlendirme
  pattern   13  desen ozet oznitelikleri                     0
  interaction 16  z ile gercek kanal profili etkilesimi      0
  cnn        -  ham bit izgarasi uzerinde kosullandirilmis CNN  0

REF (grup-goreli gorunum) YOK — Level 2'de deployment varyantini sectik,
z ekseninde de ayni tercih.

KOSULLANDIRILMIS CNN
--------------------
Ham bit desenine bakan duz bir CNN'in temel sorunu: "0101..." her ortamda
ayni seyi ifade etmiyor. Hangi RIS elemaninin onemli oldugu H_BR ve H_RU'ya
bagli. Bitlere bakan bir ag sadece desen istatistigi ogrenir (yogunluk,
puruzluluk, frekans) — ki 13 ozet oznitelik zaten onu veriyor.

Cozum FiLM: baglam vektoru (ortam + W + ust kademe tahminleri) her
konvolusyon katmanina kanal basina olcek ve kayma uretiyor.

    h = conv(x);   h = gamma(ctx) * norm(h) + beta(ctx)

Boylece ayni desen farkli ortamlarda farkli ozellik haritalari uretiyor.
CNN "desen nasil gorunuyor" degil, "bu desen bu kanalda ne yapiyor"
ogreniyor.

Tam-konvolusyonel + AdaptiveAvgPool oldugu icin dort RIS boyutunu
(8x4, 8x8, 16x8, 16x16) tek agda isliyor. Batch'te karisik boyut gelirse
kodlayici boyut basina ayri cagrilip sonuclar geri dagitiliyor.

Fiziksel port sirasi: M en hizli, sonra N, sonra polarizasyon.
Grid: (2, nRIS2, nRIS1); butun dallar physical_ris_grid kullanir.

MIMARI
------
Bir (bank, W) grubunun 358 z'si ayni ortami, ayni W'yi, ayni ust kademe
tahminlerini paylasiyor. Baglam grup basina BIR KEZ kodlanip z'lere
genisletiliyor.

Fizik dallari AYRI kodlayicilardan geciyor. Level 2'de olculdu: hepsini tek
kodlayicida birlestirmek delta RMSE'yi 0.1007'den 0.1164'e cikarmisti.

KAYIP
-----
  MSE + 0.35 * (grup ici merkezlenmis MSE); varsayilan zero ceza = 0.

Grup artik bank degil (bank, W). Hedef o grup icinde tanim geregi sifir
ortalamali.

2x2 DEGERLENDIRME IZGARASI
--------------------------
                    egitim z        holdout z
  egitim bank       (egitim)        yeni desen
  dogrulama bank    yeni geometri   ikisi de yeni

Uc hucre ayri raporlaniyor. Hangi eksende zayif oldugunu soyler.

Ayrica holdout z'lerin bir kismi egitim desenlerine cok yakin
(local_perturbation, medyan Hamming 3/512). Temiz alt kume ayri veriliyor.

ON KOSUL: Hucre 1-6.
=============================================================================
"""

import copy
import json
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import torch
import torch.nn as nn
import torch.nn.functional as F

for name in ["L1_DIR", "L2_DIR", "L3_DIR", "MODEL_DIR", "DEVICE", "FORCE_RETRAIN",
             "CHANNEL_MAP_DIR", "CHANNEL_MAP_FEATURE_NAMES"]:
    if name not in globals():
        raise RuntimeError(f"{name} tanimli degil — once kurulum hucresini kos.")

# ============================================================== ayarlar

# Z_INPUTS ve L3_RUN_TAG kurulumdan gelir; L2 RUN_TAG degiskenini ayri kullanir.
RUN_TAG = L3_RUN_TAG
SCALAR_COLUMNS = ["level1Hat", "level1StdHat"]
if L3_USE_LEVEL2_CONTEXT:
    SCALAR_COLUMNS.append("level2DeltaHat")  # sadece eski davranis ablasyonu

ARCH_VERSION = 4

SEED = 42

GROUPS_PER_BATCH = 192
Z_PER_GROUP = 48

MAX_EPOCHS = 150
PATIENCE = 20

LR = 6e-4
WEIGHT_DECAY = 1e-4

LAMBDA_CENTER = 0.35
LAMBDA_ZERO = float(L3_LAMBDA_ZERO)

CTX_DIM = 128
Z_DIM = 96

CNN_WIDTH = 48
CNN_OUT = 32

MAP_WIDTH = 16
CHANNEL_MAP_OUT = MAP_WIDTH * 2 * 3  # 2 pol x (Re, Im, genlik)

CLEAN_RATIO = 0.04

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

VALID = ["row_phys", "pattern", "interaction", "cnn", "channel_map", "covariance"]

for g in Z_INPUTS:
    if g not in VALID:
        raise RuntimeError(f"Bilinmeyen girdi: {g}. Gecerli: {VALID}")

USE_CNN = "cnn" in Z_INPUTS
USE_CHANNEL_MAP = "channel_map" in Z_INPUTS
USE_GRID = USE_CNN or USE_CHANNEL_MAP

MODEL_ROOT = MODEL_DIR / f"level3_{RUN_TAG}"
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

BEST_PT = MODEL_ROOT / "best.pt"

with open(L3_DIR / "build_manifest.json", encoding="utf-8") as f:
    MANIFEST = json.load(f)

ROW_PHYSICS = MANIFEST["rowPhysics"]
INTERACTION_NAMES = MANIFEST["interactionFeatures"]
BASE_ANALYTIC_COLUMNS = ROW_PHYSICS + INTERACTION_NAMES
COVARIANCE_NAMES = list(COVARIANCE_FEATURE_NAMES) if "covariance" in Z_INPUTS else []
ANALYTIC_COLUMNS = BASE_ANALYTIC_COLUMNS + COVARIANCE_NAMES
if COVARIANCE_NAMES:
    with open(COVARIANCE_DIR / "manifest.json", encoding="utf-8") as stream:
        covariance_manifest = json.load(stream)
    if (covariance_manifest.get("version") != COVARIANCE_VERSION
        or covariance_manifest.get("featureNames") != COVARIANCE_NAMES
        or covariance_manifest.get("usesMonteCarlo") is not False
        or covariance_manifest.get("usesTeacherTarget") is not False):
        raise RuntimeError("Kovaryans sidecar kontrati uyusmuyor.")
PATTERN_NAMES = MANIFEST["patternFeatures"]
N_Z_TOTAL = MANIFEST["nZTotal"]

CHANNEL_MAP_NAMES = list(CHANNEL_MAP_FEATURE_NAMES)
CHANNEL_MAP_VERSION = None
CHANNEL_MAP_MAX_NRIS = None
if USE_CHANNEL_MAP:
    channel_manifest_path = CHANNEL_MAP_DIR / "manifest.json"
    if not channel_manifest_path.exists():
        raise RuntimeError(f"{channel_manifest_path} yok — once kanal haritasi hucresini kos.")
    with open(channel_manifest_path, encoding="utf-8") as f:
        channel_manifest = json.load(f)
    if channel_manifest.get("featureNames") != CHANNEL_MAP_NAMES:
        raise RuntimeError("Kanal haritasi manifesti ile kod uyusmuyor.")
    if channel_manifest.get("usesMonteCarlo") or channel_manifest.get("usesTeacherTarget"):
        raise RuntimeError("Kanal haritasi sidecar kaynak kontrati bozuk.")
    CHANNEL_MAP_VERSION = channel_manifest["version"]
    CHANNEL_MAP_MAX_NRIS = int(channel_manifest["maxNRIS"])

W_COLUMNS = ["WIdx_i11", "WIdx_i12", "WIdx_i2"]

t0 = time.time()


def elapsed():
    return f"{time.time()-t0:.0f}s"


print("Device    :", DEVICE)
print("z girdileri:", Z_INPUTS)
print("Etiket    :", RUN_TAG)

# ============================================================== tablolar

GROUPS = pd.read_parquet(L3_DIR / "group_table.parquet").sort_values("groupIdx")
GROUPS = GROUPS.reset_index(drop=True)

assert (GROUPS["groupIdx"].to_numpy() == np.arange(len(GROUPS))).all()

BANKS = pd.read_parquet(L3_DIR / "bank_table.parquet")

ZMETA = pd.read_parquet(L3_DIR / "z_index_meta.parquet")

IS_TRAIN_Z = ZMETA["isTrainZ"].to_numpy(dtype=bool)

TRAIN_Z = np.nonzero(IS_TRAIN_Z)[0]
HOLD_Z = np.nonzero(~IS_TRAIN_Z)[0]

N_ZT, N_ZH = len(TRAIN_Z), len(HOLD_Z)

zpos = np.full((2, N_Z_TOTAL), -1, dtype=np.int64)
zpos[0, TRAIN_Z] = np.arange(N_ZT)
zpos[1, HOLD_Z] = np.arange(N_ZH)

IS_TRAIN_G = GROUPS["isTrainBank"].to_numpy(dtype=bool)

N_TG = int(IS_TRAIN_G.sum())
N_VG = len(GROUPS) - N_TG

assert IS_TRAIN_G[:N_TG].all() and not IS_TRAIN_G[N_TG:].any()

print(f"\nGrup (bank,W): {N_TG:,} egitim / {N_VG:,} dogrulama")
print(f"Bagimsiz bank: {GROUPS.loc[IS_TRAIN_G, 'bankID'].nunique():,} egitim / "
      f"{GROUPS.loc[~IS_TRAIN_G, 'bankID'].nunique():,} dogrulama")
print(f"z         : {N_ZT} egitim / {N_ZH} holdout")

bank_row = pd.Series(BANKS["bankRow"].to_numpy(), index=BANKS["bankID"].to_numpy())

GROUP_BANK_ROW = bank_row.reindex(GROUPS["bankID"]).to_numpy(np.int64)

assert not np.isnan(GROUP_BANK_ROW).any()

# ============================================================== satirlar


def load_rows(bank_train, z_train):
    """4 hucreden birini yukler: (bank_train, z_train) bayrak cifti."""
    prefix = "train" if bank_train else "validation"

    files = sorted((L3_DIR / "rows").glob(f"{prefix}_*.parquet"))

    if not files:
        raise RuntimeError(f"{prefix} shard'i yok")

    n_g = N_TG if bank_train else N_VG
    off = 0 if bank_train else N_TG

    n_z = N_ZT if z_train else N_ZH
    pos = zpos[0 if z_train else 1]

    Y = np.full((n_g, n_z), np.nan, dtype=np.float32)
    P = np.zeros((n_g, n_z, len(ANALYTIC_COLUMNS)), dtype=np.float32)
    seen_rows = np.zeros((n_g, n_z), dtype=bool)

    for path in files:
        D = pq.read_table(
            path,
            columns=[
                "groupIdx", "zCandidate", "isTrainZ", "targetDeltaZ",
                *BASE_ANALYTIC_COLUMNS,
            ],
        ).to_pandas()

        m = D["isTrainZ"].to_numpy(dtype=bool)

        if not z_train:
            m = ~m

        if not m.any():
            continue

        g = D["groupIdx"].to_numpy(np.int64)[m] - off
        k = pos[D["zCandidate"].to_numpy(np.int64)[m]]

        if ((g < 0) | (g >= n_g) | (k < 0) | (k >= n_z)).any():
            raise RuntimeError(f"{path.name}: L3 satir anahtari aralik disinda")
        flat = g * n_z + k
        if len(np.unique(flat)) != len(flat) or seen_rows[g, k].any():
            raise RuntimeError(f"{path.name}: tekrarli L3 grup/Z anahtari")
        seen_rows[g, k] = True
        Y[g, k] = D["targetDeltaZ"].to_numpy(np.float32)[m]
        P[g, k, :len(BASE_ANALYTIC_COLUMNS)] = D[BASE_ANALYTIC_COLUMNS].to_numpy(np.float32)[m]

    if COVARIANCE_NAMES:
        cov_files = sorted((COVARIANCE_DIR / prefix).glob("*.parquet"))
        seen_cov = np.zeros((n_g, n_z), dtype=bool)
        key_cols = ["bankID", "WIdx_i11", "WIdx_i12", "WIdx_i2"]
        lookup = pd.MultiIndex.from_frame(GROUPS[key_cols])
        for cov_path in cov_files:
            cd = pq.read_table(cov_path, columns=key_cols + ["zCandidate"] + COVARIANCE_NAMES).to_pandas()
            loc = lookup.get_indexer(pd.MultiIndex.from_frame(cd[key_cols]))
            zc = cd["zCandidate"].to_numpy(np.int64)
            if ((zc < 0) | (zc >= N_Z_TOTAL)).any():
                raise RuntimeError(f"{cov_path.name}: Z indeksi gecersiz")
            kp = pos[zc]
            keep = (loc >= off) & (loc < off + n_g) & (kp >= 0)
            cg, ck = loc[keep] - off, kp[keep]
            flat = cg * n_z + ck
            if len(np.unique(flat)) != len(flat) or seen_cov[cg, ck].any():
                raise RuntimeError(f"{cov_path.name}: tekrarli covariance anahtari")
            values = cd[COVARIANCE_NAMES].to_numpy(np.float32)[keep]
            if not np.isfinite(values).all():
                raise RuntimeError(f"{cov_path.name}: covariance NaN/Inf")
            P[cg, ck, len(BASE_ANALYTIC_COLUMNS):] = values
            seen_cov[cg, ck] = True
        if not seen_cov.all():
            raise RuntimeError(f"{prefix}: {int((~seen_cov).sum())} covariance satiri eksik")

    missing = int(np.isnan(Y).sum())

    if missing:
        raise RuntimeError(
            f"{prefix} / z_train={z_train}: {missing} hucre doldurulmadi"
        )

    return Y, P


print(f"\nSatirlar yukleniyor... [{elapsed()}]")

CELLS = {}

for bt in (True, False):
    for zt in (True, False):
        if bt and zt:
            key = "train"
        elif bt:
            key = "newZ"
        elif zt:
            key = "newGeom"
        else:
            key = "newBoth"

        CELLS[key] = load_rows(bt, zt)

        Y, P = CELLS[key]

        print(f"  {key:<9} {Y.shape}  {Y.size:,} satir")

mem = sum(P.nbytes for _, P in CELLS.values()) / 1e9

print(f"  bellek : {mem:.2f} GB")

Y_train, P_train = CELLS["train"]

print(f"\nHedef std (egitim) : {Y_train.std():.6f}")

# ============================================================== baglam girdileri

l1_all = pd.concat(
    [pd.read_csv(L1_DIR / "train.csv"), pd.read_csv(L1_DIR / "validation.csv")],
    ignore_index=True,
)

ENV_COLUMNS = ENV_FEATURES + DIRECTION_FEATURES

SCEN_COLS = ["scenario_BR", "scenario_RU"]

LOG1P_RAW = {"K_BR", "K_RU", "dist_BR", "dist_RU"}

E = l1_all[ENV_COLUMNS].to_numpy(np.float64, copy=True)

for j, name in enumerate(ENV_COLUMNS):
    if name in LOG1P_RAW:
        E[:, j] = np.log1p(np.maximum(E[:, j], 0.0))

SCEN = sorted(set(l1_all["scenario_BR"].astype(str)) | set(l1_all["scenario_RU"].astype(str)))

SC_MAP = {s: i for i, s in enumerate(SCEN)}

OH = np.zeros((len(l1_all), 2 * len(SC_MAP)))

for k, col in enumerate(SCEN_COLS):
    codes = l1_all[col].astype(str).map(SC_MAP).to_numpy()
    ok = ~pd.isna(codes)
    OH[np.nonzero(ok)[0], codes[ok].astype(np.int64) + k * len(SC_MAP)] = 1.0

env_by_bank = pd.DataFrame(
    np.concatenate([E, OH], axis=1), index=l1_all["bankID"].to_numpy()
)

ENV = env_by_bank.reindex(GROUPS["bankID"].to_numpy()).to_numpy(np.float64)

if np.isnan(ENV).any():
    raise RuntimeError("Ortam oznitelikleri eslesmedi")

N_ENV = ENV.shape[1]

# ---- W fizigi ve skalerler -------------------------------------------------

l2_all = pd.concat(
    [
        pd.read_csv(L2_DIR / "dataset_train_lsp.csv"),
        pd.read_csv(L2_DIR / "dataset_validation_lsp.csv"),
    ],
    ignore_index=True,
)

DIRECT_COLUMNS = [
    "w_traceUBR", "w_muBRPower", "w_nlosTraceUBR", "w_muBRShare", "w_nlosShare",
    "w_diagEnergyUBR", "w_diagPeakShareUBR", "w_diagCVUBR",
    "w_commonModeNormUBR", "w_polModeNormUBR", "w_modeBalanceUBR",
    "w_commonMeanModeNorm", "w_polMeanModeNorm",
    "w_s1Mean", "w_s1AbsMean", "w_s1PowerMean", "w_s1PeakShare", "w_s1CV",
    "w_s2Mean", "w_s2AbsMean", "w_s2PowerMean", "w_s2PeakShare", "w_s2CV",
    "w_s2ToS1PowerRatio", "w_s1s2Cosine",
]

l2_key = pd.MultiIndex.from_arrays(
    [l2_all["bankID"], l2_all["WIdx_i11"], l2_all["WIdx_i12"], l2_all["WIdx_i2"]]
)

g_key = pd.MultiIndex.from_arrays(
    [GROUPS["bankID"], GROUPS["WIdx_i11"], GROUPS["WIdx_i12"], GROUPS["WIdx_i2"]]
)

WPHYS = (
    pd.DataFrame(l2_all[DIRECT_COLUMNS].to_numpy(np.float64), index=l2_key)
    .reindex(g_key).to_numpy(np.float64)
)

if np.isnan(WPHYS).any():
    raise RuntimeError("W fizigi eslesmedi")

SCALARS = GROUPS[SCALAR_COLUMNS].to_numpy(np.float64)

W_IDX = GROUPS[W_COLUMNS].to_numpy(np.int64)

W_VOCAB = [int(W_IDX[:, j].max()) + 1 for j in range(3)]

print(f"\nBaglam    : {N_ENV} ortam + {len(DIRECT_COLUMNS)} W fizigi + "
      f"{SCALARS.shape[1]} skaler + W embedding")

# ============================================================== kanal haritalari

T_CHANNEL_MAPS = None

if USE_CHANNEL_MAP:
    key_columns = ["bankID", "WIdx_i11", "WIdx_i12", "WIdx_i2"]
    group_key = pd.MultiIndex.from_frame(GROUPS[key_columns])
    if group_key.has_duplicates:
        raise RuntimeError("Level 3 group anahtari tekrarli.")

    channel_maps = np.zeros(
        (len(GROUPS), len(CHANNEL_MAP_NAMES), CHANNEL_MAP_MAX_NRIS),
        dtype=np.float16,
    )
    seen_maps = np.zeros(len(GROUPS), dtype=bool)

    map_files = sorted((CHANNEL_MAP_DIR / "train").glob("*.npz"))
    map_files += sorted((CHANNEL_MAP_DIR / "validation").glob("*.npz"))
    if not map_files:
        raise RuntimeError("Kanal haritasi shard'i bulunamadi.")

    for path in map_files:
        with np.load(path, allow_pickle=False) as Z:
            if str(Z["version"].item()) != CHANNEL_MAP_VERSION:
                raise RuntimeError(f"{path.name}: kanal haritasi surumu farkli.")
            keys = Z["keys"].astype(np.int64)
            maps = Z["maps"].astype(np.float16, copy=False)
            shapes_map = Z["shapes"].astype(np.int64)

        file_key = pd.MultiIndex.from_arrays(
            [keys[:, j] for j in range(keys.shape[1])], names=key_columns
        )
        loc = group_key.get_indexer(file_key)
        # Harita sidecar'i kaynak teacher'daki 32 W'nin tamamini tasir.
        # Level 3 ise analyticValid=False olan (bank,W) gruplarini daha once
        # elemis olabilir. Bu fazla anahtarlari at; asagidaki seen_maps kontrolu
        # Level 3'te kalan her grubun yine de eksiksiz oldugunu kanitlar.
        keep = loc >= 0
        if not keep.all():
            print(f"  {path.name}: {int((~keep).sum())} L3 disinda kalan grup atlandi")
        loc = loc[keep]
        maps = maps[keep]
        shapes_map = shapes_map[keep]
        if len(loc) == 0:
            continue
        if seen_maps[loc].any():
            raise RuntimeError(f"{path.name}: tekrarli kanal haritasi grubu.")
        if maps.shape != (
            len(loc), len(CHANNEL_MAP_NAMES), CHANNEL_MAP_MAX_NRIS
        ):
            raise RuntimeError(f"{path.name}: kanal haritasi boyutu bozuk.")

        bank_rows = GROUP_BANK_ROW[loc]
        expected_shapes = BANKS.loc[
            bank_rows, ["nRIS1", "nRIS2"]
        ].to_numpy(np.int64)
        if not np.array_equal(shapes_map, expected_shapes):
            raise RuntimeError(f"{path.name}: RIS izgarasi anahtarla uyusmuyor.")

        channel_maps[loc] = maps
        seen_maps[loc] = True

    if not seen_maps.all():
        raise RuntimeError(f"{int((~seen_maps).sum())} grup icin kanal haritasi eksik.")
    if not np.isfinite(channel_maps).all():
        raise RuntimeError("Kanal haritalarinda NaN/Inf.")

    T_CHANNEL_MAPS = torch.from_numpy(channel_maps)
    print(
        f"Kanal map : {len(CHANNEL_MAP_NAMES)} harita, "
        f"{channel_maps.nbytes/1e9:.2f} GB CPU"
    )
    del channel_maps, seen_maps

# ============================================================== z girdileri

with open(L3_DIR / "z_pattern_features_physical_v2.json", encoding="utf-8") as stream:
    physical_manifest = json.load(stream)
if physical_manifest.get("version") != PATTERN_VERSION:
    raise RuntimeError("Fiziksel desen onbellegi eski; L3 veri hucresini kos.")
PAT_FEATS = np.load(L3_DIR / "z_pattern_features_physical_v2.npy")
NEAREST = np.load(L3_DIR / "z_nearest_dist.npy")

nris_row = BANKS["nRIS"].to_numpy(np.float64)

DIRTY = NEAREST / nris_row[:, None] < CLEAN_RATIO

PAT_BY_SIZE, SIZE_SHAPE, SIZE_ID, ROW_IN_SIZE = {}, {}, None, None

if USE_GRID:
    PACKED = np.load(L3_DIR / "z_patterns.npy")

    shapes = BANKS[["nRIS1", "nRIS2"]].to_numpy(np.int64)

    uniq, SIZE_ID = np.unique(shapes, axis=0, return_inverse=True)

    ROW_IN_SIZE = np.zeros(len(shapes), dtype=np.int64)

    print()

    for s in range(len(uniq)):
        sel = np.nonzero(SIZE_ID == s)[0]

        ROW_IN_SIZE[sel] = np.arange(len(sel))

        n1, n2 = int(uniq[s, 0]), int(uniq[s, 1])
        nris = 2 * n1 * n2
        nbytes = nris // 8

        bits = np.unpackbits(
            PACKED[sel][:, :, :nbytes].reshape(-1, nbytes), axis=1
        ).reshape(len(sel), N_Z_TOTAL, nris)

        PAT_BY_SIZE[s] = torch.from_numpy(bits).to(DEVICE)
        SIZE_SHAPE[s] = (n1, n2)

        print(f"  desen grubu {s}: {n1}x{n2}, {len(sel)} bank, {bits.nbytes/1e6:.0f} MB")

    del PACKED

    GROUP_SIZE_ID = SIZE_ID[GROUP_BANK_ROW]
    GROUP_ROW_IN_SIZE = ROW_IN_SIZE[GROUP_BANK_ROW]

N_ROW_IN = (
    (len(ROW_PHYSICS) if "row_phys" in Z_INPUTS else 0)
    + (len(INTERACTION_NAMES) if "interaction" in Z_INPUTS else 0)
    + (len(PATTERN_NAMES) if "pattern" in Z_INPUTS else 0)
    + len(COVARIANCE_NAMES)
)

z_desc = f"\nz seviyesi: {N_ROW_IN} skaler"
if USE_CNN:
    z_desc += f" + CNN({CNN_OUT})"
if USE_CHANNEL_MAP:
    z_desc += f" + channel-map({CHANNEL_MAP_OUT})"
print(z_desc)

# ============================================================== normalizasyon

tr = np.arange(N_TG)


def fit(X):
    mu, sd = X[tr].mean(0), X[tr].std(0)
    sd[sd < 1e-8] = 1.0
    return mu, sd


env_mu, env_sd = fit(ENV)
wph_mu, wph_sd = fit(WPHYS)
sc_mu, sc_sd = fit(SCALARS)


def T(x):
    return torch.from_numpy(np.ascontiguousarray(x.astype(np.float32)))


TENV = T((ENV - env_mu) / env_sd)
TWPH = T((WPHYS - wph_mu) / wph_sd)
TSC = T((SCALARS - sc_mu) / sc_sd)
TW = torch.from_numpy(W_IDX)

# Analitik satir girdileri: kesin pozitif sutunlarda log. Etkilesim
# oranlarinin bazilari sifir/negatif olabildigi icin otomatik olarak loglanmaz.
POS = (P_train.reshape(-1, len(ANALYTIC_COLUMNS)).min(0) > 0)
if COVARIANCE_NAMES:
    POS[len(BASE_ANALYTIC_COLUMNS):] = False

sample = np.random.default_rng(SEED).choice(N_TG, size=min(4000, N_TG), replace=False)

S = P_train[sample].reshape(-1, len(ANALYTIC_COLUMNS))
S = np.where(POS, np.log(np.maximum(S, 1e-30)), S)

rp_mu, rp_sd = S.mean(0), S.std(0)
rp_sd[rp_sd < 1e-8] = 1.0

# Normalizasyon yalniz egitim bankalari ve egitim Z'leri ile fit edilir.
train_bank_rows = np.unique(GROUP_BANK_ROW[:N_TG])
pf_flat = PAT_FEATS[train_bank_rows][:, TRAIN_Z].reshape(-1, PAT_FEATS.shape[-1])

pf_mu, pf_sd = pf_flat.mean(0), pf_flat.std(0)
pf_sd[pf_sd < 1e-8] = 1.0

TPAT = torch.from_numpy(((PAT_FEATS - pf_mu) / pf_sd).astype(np.float32)).to(DEVICE)

T_POS = torch.from_numpy(POS).to(DEVICE)
T_RP_MU = torch.from_numpy(rp_mu.astype(np.float32)).to(DEVICE)
T_RP_SD = torch.from_numpy(rp_sd.astype(np.float32)).to(DEVICE)

TPROW = torch.from_numpy(GROUP_BANK_ROW)

GAMMA_LUT = generate_ris_response_from_z(
    torch.tensor([0, 1], device=DEVICE), device=DEVICE, parity=False
)["gamma"]

del pf_flat, S

# ============================================================== model


class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout=0.08):
        super().__init__()

        self.block = nn.Sequential(
            nn.Linear(dim, dim), nn.LayerNorm(dim), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(dim, dim),
        )

        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        return F.gelu(self.norm(x + self.block(x)))


class FiLMConv(nn.Module):
    """Konvolusyon + baglama gore kanal basina olcek/kayma.
    Ayni desen farkli ortamlarda farkli ozellik haritasi uretsin diye."""

    def __init__(self, cin, cout, ctx_dim):
        super().__init__()

        self.conv = nn.Conv2d(cin, cout, 3, padding=1, bias=False)
        self.norm = nn.GroupNorm(8, cout, affine=False)
        self.film = nn.Linear(ctx_dim, 2 * cout)

        self.cout = cout

    def forward(self, x, ctx):
        h = self.norm(self.conv(x))

        gb = self.film(ctx)

        gamma, beta = gb[:, : self.cout], gb[:, self.cout :]

        return F.gelu(h * (1 + gamma[:, :, None, None]) + beta[:, :, None, None])


class ChannelMapEncoder(nn.Module):
    """Kanal haritasini her (bank,W) icin bir kez kodlar.

    Z adaylari konvolusyondan tek tek gecmez; gamma ile faz-duyarli havuzlama
    daha sonra yapilir. Bu, tam weighted CNN'e gore cok daha hizlidir.
    """

    def __init__(self, ctx_dim, cin, width=MAP_WIDTH):
        super().__init__()
        self.l1 = FiLMConv(cin, width, ctx_dim)
        self.l2 = FiLMConv(width, width, ctx_dim)

    def forward(self, x, ctx):
        return self.l2(self.l1(x, ctx), ctx)


class PatternCNN(nn.Module):
    """Tam-konvolusyonel, AdaptiveAvgPool ile sabit cikis — dort RIS boyutu
    tek agdan geciyor. Havuzlama boyut bilgisini kaybettigi icin normalize
    (n1, n2) ayrica ekleniyor."""

    def __init__(self, ctx_dim, width=CNN_WIDTH, out_dim=CNN_OUT):
        super().__init__()

        self.l1 = FiLMConv(2, width, ctx_dim)
        self.l2 = FiLMConv(width, width, ctx_dim)
        self.l3 = FiLMConv(width, width, ctx_dim)

        self.head = nn.Sequential(
            nn.Linear(2 * width + 2, 96), nn.LayerNorm(96), nn.GELU(),
            nn.Linear(96, out_dim),
        )

    def forward(self, x, ctx, n1, n2):
        h = self.l3(self.l2(self.l1(x, ctx), ctx), ctx)

        avg = F.adaptive_avg_pool2d(h, 1).flatten(1)
        mx = F.adaptive_max_pool2d(h, 1).flatten(1)

        size = torch.zeros(x.shape[0], 2, device=x.device, dtype=x.dtype)
        size[:, 0] = n1 / 16.0
        size[:, 1] = n2 / 16.0

        return self.head(torch.cat([avg, mx, size], dim=1))


class Level3Net(nn.Module):
    def __init__(self, n_env, n_wphys, n_scal, n_row, w_vocab):
        super().__init__()

        self.env_encoder = nn.Sequential(
            nn.Linear(n_env + n_scal, 192), nn.LayerNorm(192), nn.GELU(),
            nn.Dropout(0.08),
            nn.Linear(192, CTX_DIM), nn.LayerNorm(CTX_DIM), nn.GELU(),
        )

        self.wphys_encoder = nn.Sequential(
            nn.Linear(n_wphys, 96), nn.LayerNorm(96), nn.GELU(),
            nn.Linear(96, 64), nn.LayerNorm(64), nn.GELU(),
        )

        self.w_emb = nn.ModuleList(
            [nn.Embedding(v, d) for v, d in zip(w_vocab, (12, 12, 8))]
        )

        self.ctx_fusion = nn.Sequential(
            nn.Linear(CTX_DIM + 64 + 32, CTX_DIM), nn.LayerNorm(CTX_DIM), nn.GELU()
        )

        self.ctx_residual = ResidualBlock(CTX_DIM, dropout=0.08)

        self.cnn = PatternCNN(CTX_DIM) if USE_CNN else None
        self.map_encoder = (
            ChannelMapEncoder(CTX_DIM, 2 * len(CHANNEL_MAP_NAMES))
            if USE_CHANNEL_MAP else None
        )

        z_in = (
            n_row
            + (CNN_OUT if USE_CNN else 0)
            + (CHANNEL_MAP_OUT if USE_CHANNEL_MAP else 0)
        )

        self.z_encoder = (
            nn.Sequential(
                nn.Linear(z_in, Z_DIM), nn.LayerNorm(Z_DIM), nn.GELU(),
                nn.Linear(Z_DIM, Z_DIM), nn.LayerNorm(Z_DIM), nn.GELU(),
            )
            if z_in > 0
            else None
        )

        fuse_in = CTX_DIM + (Z_DIM if z_in > 0 else 0)

        self.gate = (
            nn.Sequential(nn.Linear(fuse_in, Z_DIM), nn.Sigmoid())
            if z_in > 0
            else None
        )

        self.fusion = nn.Sequential(
            nn.Linear(fuse_in, 192), nn.LayerNorm(192), nn.GELU(), nn.Dropout(0.08),
            nn.Linear(192, 96), nn.LayerNorm(96), nn.GELU(),
        )

        self.head = nn.Sequential(nn.Linear(96, 48), nn.GELU(), nn.Linear(48, 1))

    def context(self, env, scal, wphys, widx):
        e = self.env_encoder(torch.cat([env, scal], dim=1))
        p = self.wphys_encoder(wphys)
        w = torch.cat([self.w_emb[j](widx[:, j]) for j in range(3)], dim=1)

        return self.ctx_residual(self.ctx_fusion(torch.cat([e, p, w], dim=1)))

    def forward(self, ctx, zfeat):
        G = ctx.shape[0]

        if self.z_encoder is None:
            K = 1 if zfeat is None else zfeat.shape[1]
            h = self.fusion(ctx[:, None, :].expand(G, K, ctx.shape[1]))
            return self.head(h).squeeze(-1)

        K = zfeat.shape[1]

        z = self.z_encoder(zfeat)

        c = ctx[:, None, :].expand(G, K, ctx.shape[1])

        joined = torch.cat([c, z], dim=2)

        z = z * self.gate(joined)

        h = self.fusion(torch.cat([c, z], dim=2))

        return self.head(h).squeeze(-1)


model = Level3Net(
    N_ENV, len(DIRECT_COLUMNS), SCALARS.shape[1], N_ROW_IN, W_VOCAB
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Parametre : {n_params:,}")

# ============================================================== batch


def context_for(g_idx):
    return model.context(
        TENV[g_idx].to(DEVICE), TSC[g_idx].to(DEVICE),
        TWPH[g_idx].to(DEVICE), TW[g_idx].to(DEVICE),
    )


def cnn_features(g_idx, z_global, ctx):
    """Batch'te birden fazla RIS boyutu olabilir; kodlayici her boyut icin
    ayri cagrilip sonuclar geri dagitiliyor."""
    g_np = g_idx.numpy()

    G, K = len(g_np), len(z_global)

    out = torch.zeros(G, K, CNN_OUT, device=DEVICE)

    zg = z_global.to(DEVICE)

    sizes = GROUP_SIZE_ID[g_np]

    for s in np.unique(sizes):
        sel = np.nonzero(sizes == s)[0]

        rows = torch.from_numpy(GROUP_ROW_IN_SIZE[g_np[sel]]).to(DEVICE)

        bits = PAT_BY_SIZE[int(s)][rows[:, None], zg[None, :]]

        n1, n2 = SIZE_SHAPE[int(s)]

        x = physical_ris_grid(bits, n1, n2).reshape(-1, 2, n2, n1).float()

        c = ctx[torch.from_numpy(sel).to(DEVICE)]
        c = c[:, None, :].expand(len(sel), K, c.shape[1]).reshape(-1, c.shape[1])

        e = model.cnn(x, c, n1, n2)

        out[torch.from_numpy(sel).to(DEVICE)] = e.view(len(sel), K, CNN_OUT)

    return out


def channel_map_features(g_idx, z_global, ctx):
    """Harita CNN'i G kez, faz havuzlamasi GxK kez calisir."""
    g_np = g_idx.detach().cpu().numpy()
    G, K = len(g_np), len(z_global)
    out = torch.zeros(G, K, CHANNEL_MAP_OUT, device=DEVICE)
    zg = z_global.to(DEVICE)
    sizes = GROUP_SIZE_ID[g_np]

    for s in np.unique(sizes):
        sel = np.nonzero(sizes == s)[0]
        sel_t = torch.from_numpy(sel).to(DEVICE)
        global_t = torch.from_numpy(g_np[sel])

        n1, n2 = SIZE_SHAPE[int(s)]
        nris = 2 * n1 * n2

        maps = T_CHANNEL_MAPS[global_t, :, :nris].to(
            device=DEVICE, dtype=torch.float32
        )
        maps = maps.reshape(
            len(sel), len(CHANNEL_MAP_NAMES), 2, n2, n1
        ).reshape(len(sel), 2 * len(CHANNEL_MAP_NAMES), n2, n1)

        h = model.map_encoder(maps, ctx[sel_t])

        bank_rows = torch.from_numpy(GROUP_ROW_IN_SIZE[g_np[sel]]).to(DEVICE)
        bits = PAT_BY_SIZE[int(s)][bank_rows[:, None], zg[None, :]]
        gamma = physical_ris_grid(GAMMA_LUT[bits.long()], n1, n2)

        scale = float(n1 * n2)
        real_pool = torch.einsum("gchw,gkphw->gkcp", h, gamma.real) / scale
        imag_pool = torch.einsum("gchw,gkphw->gkcp", h, gamma.imag) / scale
        amp_pool = torch.sqrt(real_pool.square() + imag_pool.square() + 1e-12)

        pooled = torch.stack(
            [real_pool, imag_pool, amp_pool], dim=-1
        ).reshape(len(sel), K, CHANNEL_MAP_OUT)
        out = out.index_copy(0, sel_t, pooled)

    return out


def z_features(P_chunk, g_idx, z_global, ctx):
    parts = []

    if "row_phys" in Z_INPUTS or "interaction" in Z_INPUTS or COVARIANCE_NAMES:
        A = P_chunk.to(DEVICE)
        A = torch.where(T_POS, torch.log(A.clamp_min(1e-30)), A)
        A = (A - T_RP_MU) / T_RP_SD

        if "row_phys" in Z_INPUTS:
            parts.append(A[..., : len(ROW_PHYSICS)])

        if "interaction" in Z_INPUTS:
            parts.append(A[..., len(ROW_PHYSICS) : len(BASE_ANALYTIC_COLUMNS)])

        if COVARIANCE_NAMES:
            parts.append(A[..., len(BASE_ANALYTIC_COLUMNS):])

    if "pattern" in Z_INPUTS:
        rows = TPROW[g_idx].to(DEVICE)
        parts.append(TPAT[rows[:, None], z_global.to(DEVICE)[None, :]])

    if USE_CNN:
        parts.append(cnn_features(g_idx, z_global, ctx))

    if USE_CHANNEL_MAP:
        parts.append(channel_map_features(g_idx, z_global, ctx))

    if not parts:
        return torch.empty(len(g_idx), len(z_global), 0, device=DEVICE)

    return torch.cat(parts, dim=2)


def compute_loss(pred, true):
    base = F.mse_loss(pred, true)

    pm = pred.mean(dim=1, keepdim=True)
    tm = true.mean(dim=1, keepdim=True)

    centered = F.mse_loss(pred - pm, true - tm)

    # Sifir ortalama sadece tum 358 egitim adayi icin gecerlidir.
    # Rastgele 48 Z'lik alt kumede varsayilan olarak sifira cekme yok.
    return base + LAMBDA_CENTER * centered + LAMBDA_ZERO * pm.square().mean()


# ============================================================== degerlendirme

CELL_SPEC = {
    "newZ": (True, False),
    "newGeom": (False, True),
    "newBoth": (False, False),
}


@torch.inference_mode()
def predict_cell(key):
    model.eval()

    Y, P = CELLS[key]

    bank_train, z_train = CELL_SPEC[key]

    off = 0 if bank_train else N_TG
    n_g = Y.shape[0]

    z_global = torch.from_numpy(TRAIN_Z if z_train else HOLD_Z)

    out = np.zeros_like(Y)

    step = 256

    for s in range(0, n_g, step):
        e = min(s + step, n_g)

        g_idx = torch.from_numpy(np.arange(s, e) + off)

        ctx = context_for(g_idx)

        zf = z_features(torch.from_numpy(P[s:e]), g_idx, z_global, ctx)

        out[s:e] = model(ctx, zf).float().cpu().numpy()

    return out


def stats(pred, true):
    err = pred - true
    return {
        "RMSE": float(np.sqrt(np.mean(err**2))),
        "MAE": float(np.mean(np.abs(err))),
        "R2": float(1 - np.sum(err**2) / np.sum((true - true.mean()) ** 2)),
        "bias": float(np.mean(err)),
        "n": int(true.size),
    }


def evaluate(full=False):
    """Hizli mod: sadece newBoth. Tam mod: 2x2 izgaranin ucu."""
    keys = ["newZ", "newGeom", "newBoth"] if full else ["newBoth"]

    result = {}

    for key in keys:
        Y, _ = CELLS[key]

        pred = predict_cell(key)

        result[key] = {"all": stats(pred, Y), "pred": pred}

        bank_train, z_train = CELL_SPEC[key]

        if not z_train:
            rows = GROUP_BANK_ROW[: N_TG] if bank_train else GROUP_BANK_ROW[N_TG :]

            clean = ~DIRTY[rows][:, HOLD_Z]

            result[key]["clean"] = stats(pred[clean], Y[clean])

    return result


# ============================================================== egitim

import hashlib

def file_digest(path):
    h = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(4 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

RUN_CONTRACT = {
    "version": 4, "seed": SEED, "zInputs": Z_INPUTS,
    "scalarColumns": SCALAR_COLUMNS, "patternVersion": PATTERN_VERSION,
    "lambdaCenter": LAMBDA_CENTER, "lambdaZero": LAMBDA_ZERO,
    "groupsPerBatch": GROUPS_PER_BATCH, "zPerGroup": Z_PER_GROUP,
    "lr": LR, "weightDecay": WEIGHT_DECAY,
    "maxEpochs": MAX_EPOCHS, "patience": PATIENCE,
    "covarianceFeatures": COVARIANCE_NAMES,
    "covarianceVersion": COVARIANCE_VERSION if COVARIANCE_NAMES else None,
    "groupSHA256": file_digest(L3_DIR / "group_table.parquet"),
    "zMetaSHA256": file_digest(L3_DIR / "z_index_meta.parquet"),
    "patternSHA256": file_digest(L3_DIR / "z_pattern_features_physical_v2.npy"),
    "buildManifestSHA256": file_digest(L3_DIR / "build_manifest.json"),
}

cached = False

if BEST_PT.exists() and not FORCE_RETRAIN:
    ckpt = torch.load(BEST_PT, map_location=DEVICE, weights_only=False)

    if (
        ckpt.get("runContract") == RUN_CONTRACT
        and ckpt.get("zInputs") == Z_INPUTS
        and ckpt.get("archVersion") == ARCH_VERSION
        and ckpt.get("nEnv") == N_ENV
        and ckpt.get("interactionFeatures") == INTERACTION_NAMES
        and (
            not USE_CHANNEL_MAP
            or (
                ckpt.get("channelMapVersion") == CHANNEL_MAP_VERSION
                and ckpt.get("channelMapFeatures") == CHANNEL_MAP_NAMES
            )
        )
    ):
        model.load_state_dict(ckpt["state"])
        best_epoch = ckpt["bestEpoch"]
        history = pd.DataFrame(ckpt.get("history", []))
        cached = True

        print(f"\nMEVCUT MODEL (en iyi epoch {best_epoch}) — "
              "FORCE_RETRAIN = True ile yeniden egit")
    else:
        print("\nKaydedilmis model farkli yapilandirmayla egitilmis.")

if not cached:
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=6, min_lr=1e-6
    )

    best_rmse, best_state, best_epoch, bad = np.inf, None, -1, 0

    history = []

    rng = np.random.default_rng(SEED)

    TP = torch.from_numpy(P_train)
    TY = torch.from_numpy(Y_train)

    TRAIN_Z_T = torch.from_numpy(TRAIN_Z)

    steps = int(np.ceil(N_TG / GROUPS_PER_BATCH))

    print()
    print("=" * 78)
    print("EGITIM")
    print("=" * 78)
    print(f"Epoch basina {steps} adim, "
          f"{steps*GROUPS_PER_BATCH*Z_PER_GROUP:,} satir")
    print()

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()

        order = rng.permutation(N_TG)

        loss_sum, n_steps = 0.0, 0

        for s in range(0, N_TG, GROUPS_PER_BATCH):
            g_local = order[s : s + GROUPS_PER_BATCH]

            if len(g_local) < 8:
                continue

            g_idx = torch.from_numpy(g_local)

            zk = torch.from_numpy(
                rng.choice(N_ZT, size=Z_PER_GROUP, replace=False)
            )

            z_global = TRAIN_Z_T[zk]

            ctx = context_for(g_idx)

            zf = z_features(TP[g_idx][:, zk], g_idx, z_global, ctx)

            pred = model(ctx, zf)

            loss = compute_loss(pred, TY[g_idx][:, zk].to(DEVICE))

            optimizer.zero_grad()
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)

            optimizer.step()

            loss_sum += float(loss.item())
            n_steps += 1

        res = evaluate(full=False)["newBoth"]

        scheduler.step(res["all"]["RMSE"])

        history.append(
            {
                "epoch": epoch,
                "loss": loss_sum / max(n_steps, 1),
                "allRMSE": res["all"]["RMSE"],
                "allR2": res["all"]["R2"],
                "cleanRMSE": res["clean"]["RMSE"],
                "lr": optimizer.param_groups[0]["lr"],
            }
        )

        print(
            f"Epoch {epoch:03d} | loss={loss_sum/max(n_steps,1):.6f} | "
            f"deltaZ={res['all']['RMSE']:.6f} R2={res['all']['R2']:.4f} | "
            f"temiz={res['clean']['RMSE']:.6f} | "
            f"lr={optimizer.param_groups[0]['lr']:.1e} | {elapsed()}"
        )

        if res["all"]["RMSE"] < best_rmse - 1e-6:
            best_rmse, best_epoch, bad = res["all"]["RMSE"], epoch, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            bad += 1

        if bad >= PATIENCE:
            print(f"\nErken durdurma. En iyi epoch = {best_epoch}")
            break

    model.load_state_dict(best_state)

    history = pd.DataFrame(history)

# ============================================================== sonuc

res = evaluate(full=True)

print()
print("#" * 78)
print("EN IYI LEVEL 3 MODELI")
print("#" * 78)
print(f"Etiket      : {RUN_TAG}")
print(f"z girdileri : {Z_INPUTS}")
print(f"En iyi epoch: {best_epoch}")
print(f"Hedef std   : {Y_train.std():.6f}")

LABELS = {
    "newZ": "egitim bank + holdout z   (yeni desen)",
    "newGeom": "dogrulama bank + egitim z (yeni geometri)",
    "newBoth": "dogrulama bank + holdout z (ikisi de yeni)",
}

print()
print("2x2 IZGARA — DELTA Z")

for key in ["newZ", "newGeom", "newBoth"]:
    a = res[key]["all"]

    print(f"\n  {LABELS[key]}")
    print(f"    RMSE={a['RMSE']:.6f}  R2={a['R2']:.4f}  "
          f"MAE={a['MAE']:.6f}  bias={a['bias']:+.2e}  n={a['n']:,}")

    if "clean" in res[key]:
        c = res[key]["clean"]
        print(f"    temiz alt kume: RMSE={c['RMSE']:.6f}  R2={c['R2']:.4f}  "
              f"n={c['n']:,}")

# ---- uctan uca ------------------------------------------------------------

Yb, _ = CELLS["newBoth"]

pred = res["newBoth"]["pred"]

l2hat = GROUPS["level2Hat"].to_numpy(np.float32)[N_TG:][:, None]
wmean = GROUPS["targetWMean"].to_numpy(np.float32)[N_TG:][:, None]

y_true = wmean + Yb

e3 = (l2hat + pred) - y_true
e2 = l2hat - y_true

rows = GROUP_BANK_ROW[N_TG:]
clean = ~DIRTY[rows][:, HOLD_Z]

print()
print("UCTAN UCA (tekil konfigurasyon, dogrulama bank + holdout z)")
print(f"  Level 2 tek basina : {np.sqrt(np.mean(e2**2)):.6f}")
print(f"  Level 3            : {np.sqrt(np.mean(e3**2)):.6f}")
print(f"  R2                 : "
      f"{1 - np.sum(e3**2)/np.sum((y_true - y_true.mean())**2):.6f}")
print(f"  iyilesme           : "
      f"%{100*(1 - np.sqrt(np.mean(e3**2))/np.sqrt(np.mean(e2**2))):.1f}")

print()
print("  temiz alt kume")
print(f"  Level 2 tek basina : {np.sqrt(np.mean(e2[clean]**2)):.6f}")
print(f"  Level 3            : {np.sqrt(np.mean(e3[clean]**2)):.6f}")

# ============================================================== kayit

torch.save(
    {
        "state": model.state_dict(),
        "runContract": RUN_CONTRACT,
        "scalarColumns": SCALAR_COLUMNS,
        "patternVersion": PATTERN_VERSION,
        "covarianceFeatures": COVARIANCE_NAMES,
        "bestEpoch": best_epoch,
        "archVersion": ARCH_VERSION,
        "runTag": RUN_TAG,
        "zInputs": Z_INPUTS,
        "nEnv": N_ENV,
        "envColumns": ENV_COLUMNS,
        "scenarioMap": SC_MAP,
        "directColumns": DIRECT_COLUMNS,
        "rowPhysics": ROW_PHYSICS,
        "interactionFeatures": INTERACTION_NAMES,
        "channelMapVersion": CHANNEL_MAP_VERSION,
        "channelMapFeatures": CHANNEL_MAP_NAMES,
        "patternFeatures": PATTERN_NAMES,
        "wVocab": W_VOCAB,
        "positiveMask": POS,
        "norm": {
            "envMu": env_mu, "envSd": env_sd,
            "wphMu": wph_mu, "wphSd": wph_sd,
            "scMu": sc_mu, "scSd": sc_sd,
            "rpMu": rp_mu, "rpSd": rp_sd,
            "pfMu": pf_mu, "pfSd": pf_sd,
        },
        "metrics": {k: {m: res[k][m] for m in res[k] if m != "pred"} for k in res},
        "history": history.to_dict("records") if len(history) else [],
    },
    BEST_PT,
)

if len(history):
    history.to_csv(MODEL_ROOT / "training_history.csv", index=False)

np.save(MODEL_ROOT / "val_deltaz_pred.npy", pred)

print()
print("Kaydedildi:", MODEL_ROOT)

print()
print("ABLASYON (kurulumdaki Z_INPUTS/L3_RUN_TAG):")
print('  row_phys + pattern + interaction: interaction_fixed_v2 (varsayilan)')
print('  + covariance: interaction_covariance_v2 (8 ek skaler, ayri sidecar)')
print('  + channel_map: channel_map_fixed_v2 (istege bagli)')
print('  L3_USE_LEVEL2_CONTEXT=True: eski L2 girdisinin ablasyonu')
print('  L3_LAMBDA_ZERO=0.10: eski alt-kume cezasinin ablasyonu')
print('  Her ablasyonda farkli L3_RUN_TAG kullan; once kurulum ayarlarini uygula.')
print('  newBoth model seciminde kullanildigi icin dogrulama sonucudur.')

print(f"\nSure: {(time.time()-t0)/60:.1f} dk")
